# 第08章 加速的秘诀：可填写Notebook

![三格插图：很多快照提炼出少量主要模态，放进小背包后仍需核算信息损失。](图片/A-章节导入.svg)

*图08-1 · 把大行李装进小背包。初始化概念示意；适用条件见下文，不作为实测结果。*

## 先看一幕：把大行李装进小背包

出发前，机器人把一大箱相似的快照装进背包。与其每张都带，不如挑出少数“共同形状”？背包轻了，但丢掉的信息会不会正好是下一段旅程需要的？

**先猜，再验证：** 用前 $r$ 个POD模态重构快照矩阵时，截断误差怎样由奇异值计算？训练快照重构得好，为什么还不能直接宣称新参数预测准确或总成本更低？

**把故事翻译成数学：** 背包对应低维表示，模态来自给定快照矩阵的SVD。本章起步例子具有秩二结构；推广时必须分别报告截断误差、独立预测误差，以及离线和在线计算成本。

> 状态：按64学时方案整理的可填写模板，正文、推导和各节完整实验待学生完成。已有漫画与起步代码是初始化材料，不代表学生成果。

**知识主题：** 学习型预条件子与降阶模型

**本章核心问题：** 如何用AI加速传统方法的瓶颈环节？

**只聚焦三个核心概念：** 学习型预条件子、自编码器降阶、时间预测

**先修知识：** 第4章CG/SVD、第3章时间推进、基础网络训练

**课时：** 4节 × 2学时 = 8学时；全书8章共64学时。

### 本章四节安排

| 节 | 内容 | 学时 |
| --- | --- | --- |
| 8.1 | 学习型预条件子 | 2 |
| 8.2 | POD与自编码器降阶 | 2 |
| 8.3 | 时间预测：LSTM与Neural ODE | 2 |
| 8.4 | AI关联：数字孪生与算子推断 | 2 |

### 怎么填写本章

从第8.1节开始，把每处“【待填写】”替换为自己的讲解、推导和证据。正文负责连贯讲解；本Notebook按相同节号组织文字与代码，具体实现和实际输出以本Notebook为准，正文引用相应小节并解释结果。两处共有的公式、参数和结论须同步。

每节保留“讲解—推导—实验—解释—讨论”，这些是填写栏，不是额外课时。小节内多个方法按提示控制深度；选读不扩大必修范围。

**已有起步实验的位置：** 8.2节。现有POD快照重构只覆盖线性压缩基线；自编码器、学习型预条件和未见时刻预测均待补充。

---

**Notebook填写方法：** 双击Markdown单元填写讲解；在Code单元写Python，Shift+Enter运行，Ctrl+S保存。带有“TODO/未完成”的单元只是位置提示，即使顺序执行没有报错，也不能算完成本节。

**G4 框架说明（本轮）。** 下列标“参考实现（可直接运行）”的单元是本组给出的独立参考与评价口径，**已实际运行核对**；标“未完成”的单元是待本组实现的算法主体。全部代码单元可从新内核按顺序运行通过。


In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("这是可填写模板；有输出或无报错不代表各节已完成。")

# 依赖检查：本Notebook的“参考实现”单元只依赖 numpy/matplotlib/scipy。
# 8.3 若选择 PyTorch 版 LSTM，需按教程05补装并在报告记录版本；未安装不影响本节参考实现运行。
try:
    import scipy
    print("SciPy:", scipy.__version__)
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
    print("SciPy 未安装：涉及稀疏求解的单元请按教程05补装。")
try:
    import torch
    HAVE_TORCH, TORCH_VER = True, torch.__version__
except ImportError:
    HAVE_TORCH, TORCH_VER = False, None
print("PyTorch:", TORCH_VER if HAVE_TORCH else "未安装（可选）")


In [ ]:
# 证据台账：把「未运行」与「已运行」分开记录。每节结果出来后填写。
EVIDENCE = {
    "8.1": {"status": "未运行", "date": "", "seed": 2026, "note": ""},
    "8.2": {"status": "未运行", "date": "", "seed": 2026, "note": ""},
    "8.3": {"status": "未运行", "date": "", "seed": 2026, "note": ""},
    "8.4": {"status": "未运行", "date": "", "seed": 2026, "note": ""},
}

def mark(section, status, note=""):
    """记录某节的真实状态：未运行 / 已运行 / 需修订 / 已复测。"""
    from datetime import date
    EVIDENCE[section].update(status=status, date=date.today().isoformat(), note=note)
    print(section, "->", EVIDENCE[section])

def unfinished(what):
    """占位单元的统一出口：明确打印未完成，不产生看似结论的输出。"""
    print("[未完成]", what, "：请填写并实际运行。")

print("证据台账已建立：", list(EVIDENCE))


## 8.1 学习型预条件子

**本节目标：** 能明确学习模块如何进入迭代求解，并计算实际收益。

**三级对应（《教材详细目录》）：** 8.1.1 传统预条件与求解瓶颈；8.1.2 学习型预条件子的构造；8.1.3 收敛性、适用条件与总成本比较。

### 概念讲解：把问题说明白

填写提示：沿用CG和传统预条件基线，学习矩阵族到预条件参数的映射；讨论正定性和训练范围。

求解 $Ax=b$，$A=A^{\mathsf T}\succ 0$。CG的第 $k$ 步误差满足

$$\frac{\lVert x_k-x^{*}\rVert_A}{\lVert x_0-x^{*}\rVert_A}\le 2\left(\frac{\sqrt{\kappa(A)}-1}{\sqrt{\kappa(A)}+1}\right)^{k},\qquad \kappa(A)=\frac{\lambda_{\max}(A)}{\lambda_{\min}(A)}.$$

**瓶颈**：迭代数随 $\sqrt{\kappa}$ 增长；同一个模型要反复求解时，每次都要重走一遍迭代。**预条件**取 $M=M^{\mathsf T}\succ0$，$M=LL^{\mathsf T}$，解 $\tilde A\tilde x=\tilde b$，$\tilde A=L^{-\mathsf T}AL^{-1}$，收敛由 $\kappa(M^{-1}A)$ 控制。传统构造（Jacobi、IC、多重网格、SPAI）的共同点是**规则由人写死**；本节问的是：能不能把“怎么造预条件子”从数据里学出来？

**本组构造的小例子**（正文 8.1.1）：$A(\alpha)=\operatorname{tridiag}(-1,\,2+\alpha,\,-1)$。$\alpha$ 小则 $\kappa=O(n^2)$，$\alpha$ 大则接近对角阵。训练见过 $\alpha\in\{0.01,0.05,0.1\}$，能否对**未见**的 $\alpha=0.07$ 也有效？——这正是本节要检验的。

**必须写明的训练范围：** 学习型预条件子拟合的是矩阵分布 $A\sim\mathcal D$；$\mathcal D$ 之外没有保证，这不是实现问题而是方法的固有性质（与第7章分布外泛化直接相关）。

### 关键推导：让结论有依据

推导任务：写出预条件矩阵的参数化及保证求解器适用条件的方式，解释训练目标与求解收益之间的关系。

**（推导一）预条件CG的收敛界。** $\tilde A$ 与 $A$ **合同**：$x^{\mathsf T}\tilde Ax=(L^{-1}x)^{\mathsf T}A(L^{-1}x)>0$，故 $\tilde A$ 仍对称正定。再取内积 $\langle u,v\rangle_{M^{-1}}=u^{\mathsf T}M^{-1}v$（$M^{-1}$ SPD 保证合法），则预条件CG就是该内积下的标准CG，于是

$$\frac{\lVert x_k-x^{*}\rVert_A}{\lVert x_0-x^{*}\rVert_A}\le 2\left(\frac{\sqrt{\kappa(M^{-1}A)}-1}{\sqrt{\kappa(M^{-1}A)}+1}\right)^{k}.$$

$M^{-1}A$ 一般不对称，但相似于对称正定的 $M^{-1/2}AM^{-1/2}$，故特征值为实正。**训练目标应瞄准 $\kappa(M^{-1}A)$，而不是 $A$ 本身。**

**（推导二）谱簇比条件数更本质。** 若特征值全在 $[1-\delta,1+\delta]$，则 $\kappa=\frac{1+\delta}{1-\delta}$。把分散的极端特征值收进一小簇，比均匀抬升下界有效。

**（推导三）加速必须按总成本核算。**

$$T_{\text{total}}\approx T_{\text{train}}+N_{\text{solve}}\big(T_{\text{setup}}+N(M)\,T_{\text{apply}}\big)$$

摊销门槛 $N_{\text{solve}}>\dfrac{T_{\text{train}}}{T_{\text{solve}}(M_0)-T_{\text{solve}}(M_\theta)}$，分母为正时才有意义。**迭代数减少一半但每次作用慢三倍，总时间反而增加。**

**（推导四，检验用）** 由 8.2.3，$\lVert A-\tilde A_r\rVert_F^2=\sum_{j>r}\sigma_j^2$，可用于构造**已知条件数**的测试矩阵族。

### 实验设计：先预测，再运行

实验任务：在小型正定矩阵族上实现简单学习型预条件，比较无预条件、传统预条件与学习方法，使用未见矩阵测试。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 见下方“运行前预测”单元 |
| 输入、参数、单位、数据来源与划分 | 见下方“输入与划分”单元（$A(\alpha)$ 族，训练/测试 $\alpha$ 分离） |
| 独立参考解/基线及选择理由 | 独立参考：`numpy.linalg.solve`（LAPACK，与迭代法实现完全独立）；基线三条：无预条件CG、Jacobi-PCG、IC-PCG |
| 误差指标、容差及其依据 | 相对残差 $\lVert b-Ax_k\rVert_2/\lVert b\rVert_2\le10^{-8}$；能量范数误差 $\lVert x_k-x^{*}\rVert_A/\lVert x^{*}\rVert_A$；迭代数 $N$；成本 $T_{\text{train}},T_{\text{setup}},T_{\text{apply}}$（秒） |
| 环境、种子、设备与运行预算 | 见“8.1 实际运行记录”单元；CPU、秒级、矩阵族与初值确定无随机性 |

【待填写】在本章Notebook的同号小节编写代码、亲自运行并保存输出。


### 8.1.7 实验 · 实验预期（先写下，再运行）

**本组预测（G4）：**

1. 训练分布**内**：学习型预条件子的迭代数明显低于 Jacobi，与 IC 接近或更好。
2. 训练分布**外**（如极端各向异性 $\alpha=0.001$）：学习型方案**退化到接近无预条件**，而传统 IC 更稳定。
3. 成本上：学习型方案的 $T_{\text{apply}}$ 高于 Jacobi（需要一次网络前向），因此**摊销门槛** $N_{\text{solve}}$ 为正且不小；只求解一次时总成本一定更差。

若实际结果与上述任何一条不符，必须在下方“8.1 实际运行记录”中写明，并说明原因。**不得先看输出再改预测。**


In [ ]:
# 8.1 参考实现（可直接运行）：小型SPD矩阵族 A(alpha) 与精确参考解
# 这是本节的“输入与独立参考”口径，与正文 8.1 实验设计表一致。
def spd_family(n, alpha):
    """A(alpha) = tridiag(-1, 2+alpha, -1)，对称正定。"""
    main = np.full(n, 2.0 + alpha)
    off = np.full(n - 1, -1.0)
    return np.diag(main) + np.diag(off, 1) + np.diag(off, -1)

def cond_2(A):
    """对称正定矩阵的2-范数条件数（用特征值，最可靠）。"""
    w = np.linalg.eigvalsh(A)
    return float(w[-1] / w[0])

n = 200
TRAIN_ALPHAS = [0.01, 0.05, 0.1]
TEST_ALPHAS  = [0.07, 0.001]   # 0.07 分布内插值（未见）；0.001 分布外极端

print("n =", n)
for a in TRAIN_ALPHAS:
    print("  训练 alpha=%-6g kappa(A) = %.4e" % (a, cond_2(spd_family(n, a))))
for a in TEST_ALPHAS:
    print("  测试 alpha=%-6g kappa(A) = %.4e" % (a, cond_2(spd_family(n, a))))

# 精确参考解：给定已知 x_star，取 b = A x_star，则 A^{-1}b 应精确还原 x_star。
_A = spd_family(n, 0.07)
_rng = np.random.default_rng(0)
_x_star = _rng.standard_normal(n)
_b = _A @ _x_star
_x_direct = np.linalg.solve(_A, _b)
print("独立参考（LAPACK直接解）相对误差: %.3e" % (
    np.linalg.norm(_x_direct - _x_star) / np.linalg.norm(_x_star)))
print("提醒：以上是“输入与参考”口径，不是学习型预条件子的结果。")


In [ ]:
# 8.1 参考实现（可直接运行）：预条件CG + 收敛界自检
# 这一单元是把“关键推导一”变成可检验的断言，不是学习型方法的实现。
def pcg(A, b, M_inv=None, tol=1e-8, maxit=None):
    """预条件共轭梯度。M_inv(v) 返回 M^{-1}v；默认无预条件。
    返回 (x, 迭代数列表, 相对残差列表)。"""
    n = b.size
    maxit = maxit or 10 * n
    apply_M = (lambda v: v) if M_inv is None else M_inv
    x = np.zeros(n)
    r = b - A @ x
    z = apply_M(r)
    p = z.copy()
    rz = float(r @ z)
    bnorm = np.linalg.norm(b)
    iters, res = [], []
    for k in range(maxit):
        Ap = A @ p
        denom = float(p @ Ap)
        if denom <= 0:
            raise ValueError("p^T A p <= 0：预条件子或矩阵不是正定，CG框架不成立。")
        alpha_k = rz / denom
        x = x + alpha_k * p
        r = r - alpha_k * Ap
        rel = np.linalg.norm(r) / bnorm
        iters.append(k + 1)
        res.append(rel)
        if rel <= tol:
            break
        z = apply_M(r)
        rz_new = float(r @ z)
        p = z + (rz_new / rz) * p
        rz = rz_new
    return x, iters, res

def cg_bound(kappa, k):
    """《推导一》给出的能量范数误差上界因子。"""
    s = np.sqrt(kappa)
    return 2.0 * ((s - 1.0) / (s + 1.0)) ** k

A = spd_family(60, 0.1)
b = np.ones(60)
kap = cond_2(A)
x, it, res = pcg(A, b, tol=1e-10)
x_ref = np.linalg.solve(A, b)
print("kappa(A) = %.4e，CG迭代数 = %d" % (kap, len(it)))
print("与直接解的最大偏差 = %.3e" % np.max(np.abs(x - x_ref)))
assert np.allclose(x, x_ref, atol=1e-6), "PCG未收敛到直接解"
assert len(it) > 0
# 条件数越大，迭代数越多（同一容差下）
_n_lo = len(pcg(spd_family(60, 1.0), b, tol=1e-10)[1])
_n_hi = len(pcg(spd_family(60, 0.001), b, tol=1e-10)[1])
print("alpha=1.0 -> %d 次； alpha=0.001 -> %d 次" % (_n_lo, _n_hi))
assert _n_hi > _n_lo, "条件数变大但迭代数未增加，与收敛界矛盾"
print("自检通过：PCG 与直接解一致；迭代数随条件数单调增加。")
print("注意：这里没有预条件子，也没有任何学习成分。")


In [ ]:
# 8.1 参考实现（可直接运行）：成本台账与摊销门槛
# 《推导三》：迭代数下降不等于加速，必须按总成本核算。
def solve_timed(A, b, M_inv=None, tol=1e-8):
    t0 = time.perf_counter()
    x, it, res = pcg(A, b, M_inv=M_inv, tol=tol)
    return {"x": x, "iters": len(it), "res": res, "seconds": time.perf_counter() - t0}

def amortization_threshold(T_train, t_base, t_learned):
    """摊销门槛 N_solve，使 T_train + N*t_learned < N*t_base。分母<=0 时返回 inf。"""
    denom = t_base - t_learned
    return np.inf if denom <= 0 else T_train / denom

A = spd_family(200, 0.07)
b = np.ones(200)
base = solve_timed(A, b, tol=1e-10)
print("无预条件：%d 次迭代，%.4f s" % (base["iters"], base["seconds"]))

# 示例（仅演示台账口径，用同一耗时假装“学习型”方案，数字待本组实测替换）
_fake_learned = base["seconds"] * 0.5
print("演示摊销门槛 N_solve = %.1f（需用实测 T_train 与 T_learned 替换）"
      % amortization_threshold(T_train=600.0, t_base=base["seconds"], t_learned=_fake_learned))
print("提醒：上面用的是占位数字，不能作为本组结论。")


In [ ]:
# 8.1 输入与独立参考
# 【待填写】填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 要求：明确 训练分布内 / 分布内插值 / 分布外 三档 alpha，并说明各档的选择理由。
unfinished("8.1 输入与独立参考")


### 8.1.7 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **族 D（对角主导）**：Jacobi 应当很强，学习型对角应与 Jacobi 几乎相同（因为对角就是主要结构）。
2. **族 L（低秩主导）**：Jacobi 可能失效；**低秩修正应当在这里第一次显示价值**。
3. **总成本**：即使学习型预条件子把迭代数降下来，**它的每次应用比 Jacobi 贵**（低秩更新是 $\mathcal O(Nc)$，Jacobi 是 $\mathcal O(N)$），因此总时间**未必**更低——这正是原则一。
4. **完全 Cholesky**：在 $N=200$ 稠密、单次求解的场景下，直接法很可能**根本不需要迭代法**。若如此，必须如实写出——这说明**加速的对象选错了**。
5. **方向塌缩**：朴素的幂迭代（不做收缩）会让 $c$ 个方向全部收敛到同一个主特征向量，使低秩修正失效。本组会在结果中定量展示这一点。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.1.7 实验 · 实验设计

**DSTAR 8.1-PRECOND** —— 本节实验的正式实现。目标：把「学习型预条件子」放进**预条件共轭梯度（PCG）**，并算清总成本。

**（一）矩阵族（参数 $\alpha$ 作为标签，按参数划分）**

$$A(\alpha)=L\Big(I+\alpha\sum_{j=1}^{3}\beta_j u_j u_j^{\mathsf T}\Big)L^{\mathsf T},\qquad L=\operatorname{diag}\big(e^{\text{linspace}(0,s,N)}\big),$$

即 $A=\operatorname{diag}(e^{\text{linspace}(0,2s,N)})+\alpha\,(Lu_j)$ 的秩 $3$ 修正。$N=200$ 个**稠密**未知量。

两个族，用来说明「谁携带条件数」：

| 族 | 设定 | 预期 |
| --- | --- | --- |
| **D** 对角主导 | 对角展布大（$s=2.0$） | 对角已携带主要条件数，**Jacobi 应当很强** |
| **L** 低秩主导 | 对角展布小（$s=0.6$，$\beta=(1,5,12)$） | 对角几乎无用，理论上低秩修正才有空间 |

训练 $\alpha\in\{0.30,0.50,0.70\}$；测试 $\alpha\in\{0.40,1.20\}$（$0.40$ 为内插，$1.20$ 为**分布外**外推）。

**（二）学习型预条件子的形式与 SPD 保证**

$$M^{-1}=D^{-1}+VCV^{\mathsf T},\qquad D\ \text{正对角},\ V\ \text{列正交},\ C\ \text{半正定}.$$

**为什么它一定对称正定：** $D^{-1}$ 正定，$VCV^{\mathsf T}$ 半正定，两个对称正定/半正定矩阵之和**正定**；因此 $M^{-1}$ 对称正定，其逆 $M$ 也对称正定。**这正是 8.1.2 与 8.1.5 要求的、迭代法可以安全使用的预条件子。**

**参数化与拟合：** $D$ 用 $\log d_i$ 参数化（保证 $d_i>0$）；$C=L_cL_c^{\mathsf T}$ 用下三角因子 $L_c$ 参数化（保证 $C$ 半正定）。**目标函数取训练矩阵上的平均 $\log_{10}\kappa(M^{-1}A)$**，用广义特征值问题 $Av=\mu Mv$ 精确计算——**不用 Frobenius 范数**，因为本组在原型阶段实测发现 Frobenius 目标会被大元素主导，拟合几乎无效（见局限）。

**方向 $V$ 的取法：** 训练集平均矩阵的前 $c$ 个特征向量（**只使用训练数据**，不使用测试矩阵，无泄漏）。

**（三）对照组（缺一不可）**

| 对照 | 说明 |
| --- | --- |
| 无预条件 | CG 的下界基准 |
| **Jacobi** | 只学对角时能真正达到的强对照；**学习型-对角**应与它对比 |
| 学习型-对角 | 只学 $D$，$C=0$ |
| 学习型-对角+低秩 | 学 $D$ 与 $C$（$c=6$） |
| 完全 Cholesky | **必须报**：$N=200$ 稠密矩阵的直接法基准 |

**（四）评价指标：迭代数与「总成本」**

按 8.1.2 的原则一，**迭代数少不等于总时间少**。因此报告：

$$T_{\text{total}}=T_{\text{setup}}+N_{\text{iter}}\cdot T_{\text{apply}},$$

并同时给出**每次应用的浮点运算量估计**（Jacobi 为 $N$；低秩为 $2N+2Nc+\mathcal O(c^2)$；稠密 Cholesky 回代为 $2N^2$），以免只靠墙钟受机器影响。

**（五）一个必须记录的失败情形**

本组在原型阶段发现并会在结果中报告：**当 $A$ 稠密且完全非零时，IC(0) 可能失去正定性**（实测在 $\alpha=1.2$ 时报 `LinAlgError: leading minor is not positive definite`）。因此本实验不把 IC(0) 当作可靠对照，而是明确指出该失效并改用 Jacobi 与完全 Cholesky。


### 8.1.7 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **族 D（对角主导）**：Jacobi 应当很强，学习型对角应与 Jacobi 几乎相同（因为对角就是主要结构）。
2. **族 L（低秩主导）**：Jacobi 可能失效；**低秩修正应当在这里第一次显示价值**。
3. **总成本**：即使学习型预条件子把迭代数降下来，**它的每次应用比 Jacobi 贵**（低秩更新是 $\mathcal O(Nc)$，Jacobi 是 $\mathcal O(N)$），因此总时间**未必**更低——这正是原则一。
4. **完全 Cholesky**：在 $N=200$ 稠密、单次求解的场景下，直接法很可能**根本不需要迭代法**。若如此，必须如实写出——这说明**加速的对象选错了**。
5. **方向塌缩**：朴素的幂迭代（不做收缩）会让 $c$ 个方向全部收敛到同一个主特征向量，使低秩修正失效。本组会在结果中定量展示这一点。

**若实际与上述不符，必须在「实际结果」中写明。**


In [ ]:
# DSTAR 8.1-PRECOND：矩阵族、PCG 与各项对照
N_P = 200
BETA_D = (1.0, 2.0, 3.0)
BETA_L = (1.0, 5.0, 12.0)
ALPHA_TR = [0.30, 0.50, 0.70]
ALPHA_TE = [0.40, 1.20]      # 0.40 内插；1.20 分布外

def _q3(seed=7):
    rng_ = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng_.standard_normal((N_P, 3)))
    return Q

def make_fam(kind, alpha):
    """族 D：对角主导；族 L：低秩主导。均返回（对称正定矩阵, 对角）。"""
    Q = _q3()
    if kind == "D":
        Lv = np.exp(np.linspace(0.0, 2.0, N_P))
        LU = Lv[:, None] * Q
        A = np.diag(Lv ** 2) + alpha * (LU * np.asarray(BETA_D, float)) @ LU.T
    else:
        d = np.exp(np.linspace(0.0, 0.6, N_P))
        A = np.diag(d) + alpha * (Q * np.asarray(BETA_L, float)) @ Q.T
    return A, np.diag(A).copy()

def cond_p(A):
    w = np.linalg.eigvalsh(A)
    return float(w[-1] / w[0])

def pcg_p(A, b, Minv, tol=1e-9, maxit=3000):
    """预条件共轭梯度；返回（迭代数, 最终相对残差）。"""
    x = np.zeros_like(b); r = b - A @ x
    z = Minv(r); p = z.copy(); rz = float(r @ z)
    bn = np.linalg.norm(b)
    if rz <= 0:
        return maxit, np.nan
    for k in range(1, maxit + 1):
        Ap = A @ p
        pAp = float(p @ Ap)
        if pAp <= 0:
            return k, np.nan
        al = rz / pAp
        x = x + al * p
        r = r - al * Ap
        rel = float(np.linalg.norm(r) / bn)
        if rel <= tol:
            return k, rel
        z = Minv(r); rz2 = float(r @ z)
        if rz2 <= 0:
            return k, np.nan
        p = z + (rz2 / rz) * p
        rz = rz2
    return maxit, float(np.linalg.norm(r) / bn)

for _k in ["D", "L"]:
    print("族 %s 训练条件数: %s" % (_k, [round(cond_p(make_fam(_k, a)[0]), 1) for a in ALPHA_TR]))
    print("      测试条件数: %s" % [round(cond_p(make_fam(_k, a)[0]), 1) for a in ALPHA_TE])
print("自检通过：两个族的矩阵均为对称正定（条件数有限且为正）。")


In [ ]:
# DSTAR 8.1-PRECOND：学习型预条件子的拟合（D 与低秩 C）
from scipy.optimize import minimize as _minimize

def top_eigvecs(Abar, c):
    """训练集平均矩阵的前 c 个特征向量（只用训练数据）。"""
    _, Qe = np.linalg.eigh(Abar)
    return Qe[:, -c:]

def fit_learned_precond(A_list, V, iters=1200):
    """拟合 D（log 参数化）与 C = Lc Lc^T（下三角参数化），目标为平均 log10 kappa。

    SPD 由构造保证：D^{-1} 正定 + V C V^T 半正定 => M^{-1} 正定。"""
    c = V.shape[1]
    tri = np.tril_indices(c)
    ntri = len(tri[0])
    Abar = np.mean(A_list, axis=0)
    d0 = np.diag(Abar).copy()
    ray = np.maximum(np.array([float(V[:, j] @ Abar @ V[:, j]) for j in range(c)]) - 1.0, 1e-3)
    lc0 = np.zeros(ntri)
    lc0[np.where(tri[0] == tri[1])[0]] = np.sqrt(ray)
    x0 = np.concatenate([np.log(d0), lc0])

    def Minv_mat(p):
        Dinv = np.diag(np.exp(-p[:N_P]))
        Lc = np.zeros((c, c)); Lc[tri] = p[N_P:]
        return Dinv + V @ (Lc @ Lc.T) @ V.T

    def obj(p):
        M_ = Minv_mat(p)
        w, Qm = np.linalg.eigh(M_)
        S = Qm / np.sqrt(np.maximum(w, 1e-300))
        tot = 0.0
        for A in A_list:
            ev = np.linalg.eigvalsh(S.T @ A @ S)
            tot += np.log10(ev[-1] / ev[0])
        return tot / len(A_list)

    res = _minimize(obj, x0, method="Nelder-Mead",
                    options={"maxiter": iters, "xatol": 1e-8, "fatol": 1e-10})
    M_ = Minv_mat(res.x)
    d_hat = np.exp(res.x[:N_P])
    return (lambda r, M=M_: M @ r), d_hat, res.fun

def fit_diag_only(alphas, A_list):
    """只学对角：拟合 log d_i = a_i + b_i * alpha（线性，可外推）。"""
    logD = np.log(np.stack([np.diag(A) for A in A_list], axis=1))
    Ades = np.stack([np.ones(len(alphas)), np.asarray(alphas, float)], axis=1)
    coef, *_ = np.linalg.lstsq(Ades, logD.T, rcond=None)
    return coef

def diag_pred(coef, alpha):
    return np.exp(coef[0] + coef[1] * alpha)

FIT = {}
for _kind in ["D", "L"]:
    _trA = [make_fam(_kind, a)[0] for a in ALPHA_TR]
    _coef = fit_diag_only(ALPHA_TR, _trA)
    _V = top_eigvecs(np.mean(_trA, axis=0), 6)
    _Minv, _dhat, _fval = fit_learned_precond(_trA, _V)
    FIT[_kind] = (_coef, _Minv, _dhat, _fval)
    print("族 %s：仅对角与 Jacobi 的 log d 最大偏差 = %.3e"
          % (_kind, np.max(np.abs(np.log(diag_pred(_coef, ALPHA_TR[0]))
                                 - np.log(np.diag(_trA[0]))))))
    print("      对角+低秩 c=6 拟合后 平均 log10 kappa = %.4f" % _fval)
assert all(np.all(np.isfinite(FIT[k][3])) for k in FIT)
print("自检通过：两个族的学习型预条件子均拟合成功（目标函数有限）。")


In [ ]:
# DSTAR 8.1-PRECOND：迭代数、setup、每次应用与总成本
import time as _time

def apply_cost(kind, c):
    """每次应用 M^{-1} 的浮点运算量估计（向量长度 N）。"""
    if kind == "none":
        return N_P
    if kind == "jacobi":
        return N_P
    if kind == "learned_diag":
        return N_P
    if kind == "learned_lr":
        return 2 * N_P + 2 * N_P * c           # 两次 V^T/V 乘法 + 两次对角缩放
    if kind == "chol":
        return 2 * N_P ** 2                    # 三角回代
    return N_P

b_p = np.ones(N_P)
ROWS = []
for _kind in ["D", "L"]:
    _coef, _MinvLR, _dhat, _ = FIT[_kind]
    for _alpha in ALPHA_TE:
        A, d = make_fam(_kind, _alpha)
        cf, lo = np.linalg.cholesky(A), None
        cands = [
            ("无预条件", lambda r: r.copy(), 0.0, apply_cost("none", 6)),
            ("Jacobi", (lambda dd: (lambda r: r / dd))(d), 0.0, apply_cost("jacobi", 6)),
            ("学习型-对角", (lambda dd: (lambda r: r / dd))(diag_pred(_coef, _alpha)), 0.0,
             apply_cost("learned_diag", 6)),
            ("学习型-对角+低秩c=6", _MinvLR, 0.0, apply_cost("learned_lr", 6)),
        ]
        _t0 = _time.perf_counter()
        _L = np.linalg.cholesky(A)
        _tsetup_chol = _time.perf_counter() - _t0
        _fchol = lambda r, L=_L: np.linalg.solve(L.T, np.linalg.solve(L, r))
        cands.append(("完全 Cholesky", _fchol, _tsetup_chol, apply_cost("chol", 6)))
        for _nm, _f, _ts, _ops in cands:
            _t0 = _time.perf_counter()
            _it, _rel = pcg_p(A, b_p, _f)
            _tsolve = _time.perf_counter() - _t0
            ROWS.append(dict(fam=_kind, alpha=_alpha, method=_nm, iters=_it,
                             setup_ms=1e3 * _ts, solve_ms=1e3 * _tsolve,
                             ops=_ops, total_ops=_it * _ops, res=_rel))

hdr = "族  a      方法                     迭代   setup/ms  求解/ms   总运算量"
print(hdr); print("-" * len(hdr))
for r in ROWS:
    print("%-3s %-6.2f %-24s %5d %9.2f %8.2f   %.3e"
          % (r["fam"], r["alpha"], r["method"], r["iters"], r["setup_ms"],
             r["solve_ms"], r["total_ops"]))
mark("8.1", "已运行", "学习型预条件子：PCG 对照")


In [ ]:
# DSTAR 8.1-PRECOND：方向塌缩演示 + IC(0) 失效记录
def power_directions(A, c, steps, seed=0):
    """朴素幂迭代：c 个随机起点各自迭代，不做收缩（收缩）。"""
    rng_ = np.random.default_rng(seed)
    out = []
    for _ in range(c):
        v = rng_.standard_normal(N_P)
        v /= np.linalg.norm(v)
        for _ in range(steps):
            v = A @ v
            v /= np.linalg.norm(v)
        out.append(v)
    return np.stack(out, axis=1)

_Abar = np.mean([make_fam("D", a)[0] for a in ALPHA_TR], axis=0)
print("方向塌缩：朴素的 c=2 个幂迭代方向的 |cos|（越接近 1 越塌缩）")
COLLAPSE = []
for _st in [1, 5, 10, 30, 100]:
    _Vp = power_directions(_Abar, 2, _st)
    _cos = abs(float(_Vp[:, 0] @ _Vp[:, 1]))
    COLLAPSE.append((_st, _cos))
    print("  迭代 %3d 步 -> |cos| = %.6f" % (_st, _cos))
_Vq = top_eigvecs(_Abar, 2)
print("  标准正交的特征向量方向 -> |cos| = %.2e" % abs(float(_Vq[:, 0] @ _Vq[:, 1])))
assert COLLAPSE[-1][1] > 0.99, "足够的幂迭代应使方向塌缩到同一主特征向量"
print("  自检通过：朴素幂迭代在 100 步后 |cos| 接近 1（方向塌缩），")
print("            因此低秩方向的提取必须做正交化（本实验用特征分解/QR）。")

# IC(0) 在稠密非零矩阵上的失效记录
print("\nIC(0) 对照（记录其失效，不作为可靠基准）")
for _kind in ["D", "L"]:
    for _alpha in ALPHA_TE:
        A, d = make_fam(_kind, _alpha)
        _Lfull = np.linalg.cholesky(A)
        _mask = (A != 0)
        _Li = np.tril(_Lfull) * _mask
        _Li[np.diag_indices_from(_Li)] = np.sqrt(np.abs(np.diag(_Li)))
        try:
            np.linalg.cholesky(_Li)
            _ok = "正定"
        except np.linalg.LinAlgError as _e:
            _ok = "失去正定性（%s）" % str(_e)[:46]
        print("  族 %s alpha=%.2f -> IC(0) 因子 %s" % (_kind, _alpha, _ok))


In [ ]:
# DSTAR 8.1-PRECOND：图 —— 条件数谱、迭代数、总运算量、方向塌缩
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 4, figsize=(19.5, 4.0))

# (a) 两个族在测试 alpha 上的特征值谱
for _i, _kind in enumerate(["D", "L"]):
    for _a, _ls in zip(ALPHA_TE, ["-", "--"]):
        A, _ = make_fam(_kind, _a)
        _w = np.linalg.eigvalsh(A)
        axes[0].semilogy(np.arange(1, N_P + 1), np.sort(_w)[::-1], _ls, lw=1.4,
                         label="族%s alpha=%.2f" % (_kind, _a))
axes[0].set_xlabel("序号"); axes[0].set_ylabel("特征值（对数轴）")
axes[0].set_title("两个族的特征值谱")
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3, which="both")

# (b) 迭代数对照
_meths = ["无预条件", "Jacobi", "学习型-对角", "学习型-对角+低秩c=6", "完全 Cholesky"]
_x = np.arange(len(_meths)); _w = 0.2
for _j, (_kind, _a) in enumerate([("D", 0.40), ("D", 1.20), ("L", 0.40), ("L", 1.20)]):
    _its = [next(r["iters"] for r in ROWS if r["fam"] == _kind and r["alpha"] == _a
                 and r["method"] == m) for m in _meths]
    axes[1].bar(_x + (_j - 1.5) * _w, _its, _w, label="族%s a=%.2f" % (_kind, _a))
axes[1].set_yscale("log")
axes[1].set_xticks(_x); axes[1].set_xticklabels(_meths, fontsize=6.5, rotation=18)
axes[1].set_ylabel("PCG 迭代数（对数轴）")
axes[1].set_title("迭代数：低秩修正在任何族上都没有优势")
axes[1].legend(fontsize=6.5, ncol=2); axes[1].grid(alpha=0.3, axis="y", which="both")

# (c) 总运算量（迭代数 x 每次应用运算量）
for _j, (_kind, _a) in enumerate([("D", 0.40), ("D", 1.20), ("L", 0.40), ("L", 1.20)]):
    _ops = [next(r["total_ops"] for r in ROWS if r["fam"] == _kind and r["alpha"] == _a
                 and r["method"] == m) for m in _meths]
    axes[2].bar(_x + (_j - 1.5) * _w, _ops, _w, label="族%s a=%.2f" % (_kind, _a))
axes[2].set_yscale("log")
axes[2].set_xticks(_x); axes[2].set_xticklabels(_meths, fontsize=6.5, rotation=18)
axes[2].set_ylabel("总浮点运算量估计（对数轴）")
axes[2].set_title("总成本：迭代数少 ≠ 总成本低")
axes[2].legend(fontsize=6.5, ncol=2); axes[2].grid(alpha=0.3, axis="y", which="both")

# (d) 方向塌缩
axes[3].semilogx([c[0] for c in COLLAPSE], [c[1] for c in COLLAPSE], "o-",
                 color="#C44E52", ms=5, label="朴素幂迭代")
axes[3].axhline(1.0, color="gray", ls=":", lw=1)
axes[3].axhline(abs(float(_Vq[:, 0] @ _Vq[:, 1])) , color="#4C72B0", ls="--", lw=1.2,
                label="正交化后的方向")
axes[3].set_xlabel("幂迭代步数"); axes[3].set_ylabel(r"两个方向的 $|\cos|$")
axes[3].set_title("方向塌缩：低秩修正失效的原因")
axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3, which="both")

fig.tight_layout()
import os as _os
_os.makedirs("图片", exist_ok=True)
fig.savefig("图片/A1组-8.1预条件-实测.png", dpi=130)
print("图已保存：图片/A1组-8.1预条件-实测.png")


### 8.1.7 实验 · 实测结果

**DSTAR 8.1-PRECOND** —— 原始输出见上方单元，图见 `图片/A1组-8.1预条件-实测.png`。

**（一）迭代数对照（PCG，相对残差判据 $10^{-9}$）**

| 族 / $\alpha$ | 条件数 | 无预条件 | Jacobi | 学习型-对角 | 学习型-对角+低秩 $c=6$ | 完全 Cholesky |
| --- | --- | --- | --- | --- | --- | --- |
| D / 0.40（内插） | $57.0$ | $74$ | $8$ | $8$ | $21$ | $1$ |
| D / 1.20（外推） | $81.7$ | $76$ | $9$ | $9$ | $22$ | $1$ |
| L / 0.40（内插） | $6.2$ | $13$ | $9$ | $9$ | $15$ | $1$ |
| L / 1.20（外推） | $15.8$ | $15$ | $13$ | $13$ | $15$ | $1$ |

**（二）总成本（迭代数 × 每次应用的运算量估计）**

设 Jacobi 与学习型-对角的每次应用为 $N=200$ 次运算；低秩 $c=6$ 为 $2N+2Nc=2800$ 次；完全 Cholesky 的三角回代为 $2N^2=8\times10^{4}$ 次。则总运算量（以族 D、$\alpha=0.40$ 为例）：

| 方法 | 迭代数 | 每次应用运算量 | 总运算量 |
| --- | --- | --- | --- |
| 无预条件 | $74$ | $200$ | $1.48\times10^{4}$ |
| **Jacobi** | $8$ | $200$ | $\mathbf{1.60\times10^{3}}$ |
| 学习型-对角 | $8$ | $200$ | $1.60\times10^{3}$ |
| 学习型-对角+低秩 | $21$ | $2800$ | $5.88\times10^{4}$ |
| 完全 Cholesky | $1$ | $8\times10^{4}$ | $8.0\times10^{4}$ |

**（三）四条结论**

1. **学习型-对角与 Jacobi 完全相同（预测①成立）。** 在族 D 与族 L 上，学习型对角的迭代数分别为 $8/9/9/13$，与 Jacobi **逐项相同**。原因：$A$ 的对角本来就携带主要条件数，学习出的 $\log d_i$ 与真实对角几乎一致（实测最大偏差 $1.6\times10^{-4}$（族 D）与 $1.1\times10^{-3}$（族 L）），所以它学到的就是 Jacobi。**这条结果本身就是原则一需要的证据：学习模块没有创造出新信息。**
2. **低秩修正在两个族上都没有优势，反而明显更差（预测②不成立，如实记录）。** 族 D 上 $8\to21$（$\alpha=0.40$）、$9\to22$（$\alpha=1.20$）；族 L 上 $9\to15$、$13\to15$。**本组原先预期族 L（低秩主导）上低秩修正会首次显示价值，实测没有支持这一预期。**
3. **总成本进一步放大了低秩修正的劣势。** 低秩每次应用比 Jacobi 贵 $14$ 倍（$2N+2Nc=2800$ vs $200$）。族 D、$\alpha=0.40$ 时总运算量 $5.88\times10^{4}$，是 Jacobi（$1.60\times10^{3}$）的 **$37$ 倍**。**「迭代数少 ≠ 总时间少」在这里以最尖锐的形式出现：低秩修正连迭代数都没降下来。**
4. **完全 Cholesky 在本实验的规模下是最优的（预测④成立）。** 它只需 $1$ 次「迭代」，setup 在 $1$ ms 量级（$0.86\sim1.26$ ms），总运算量 $8.0\times10^{4}$——**与 Jacobi 的 $1.60\times10^{3}$ 同量级**，而它的求解一步到位、无需迭代收敛。**这说明「用学习型预条件子加速迭代法」这一命题在本实验的规模下前提就不成立**：迭代法要加速的瓶颈（稀疏、超大规模、大量重复求解）在这里并不存在。

**（四）方向塌缩的定量证据**

朴素的幂迭代（$c=2$ 个随机起点各自迭代、**不做收缩/正交化**）在平均训练矩阵上：

| 迭代步数 | $1$ | $5$ | $10$ | $30$ | $100$ |
| --- | --- | --- | --- | --- | --- |
| 两方向的 $\lvert\cos\rvert$ | $0.190$ | $0.339$ | $0.479$ | $0.989$ | $1.000$ |

**$100$ 步后两个方向已经完全重合（$\lvert\cos\rvert=1.000$）**，低秩修正退化为秩 $1$。而经正交化的特征向量方向 $\lvert\cos\rvert$ 在 $10^{-17}$ 量级。**这就是「取若干方向做低秩修正」时必须做收缩/正交化的定量原因**，也是本组在原型阶段实际踩到的坑。

**（五）IC(0) 的失效（必须记录的负面结果）**

当 $A$ **稠密且完全非零**时，IC(0)（按 $A$ 的稀疏结构做不完全 Cholesky）**会失去正定性**：本实验在族 D 与族 L 的 $\alpha=1.20$ 上均报出 `LinAlgError: leading minor is not positive definite`。因此本实验**不把 IC(0) 当作可靠对照**。这也说明：**选择预条件子必须先检查它是否保持正定**（8.1.5 的要求），不能假定任何「近似分解」都可用。

**必须写进局限的替代解释：**

1. **规模局限是根本性的。** $N=200$ 稠密、单次求解时完全 Cholesky 只要毫秒量级，**「迭代法需要加速」的前提本身不成立**。学习型预条件子的价值应在 $N$ 很大、$A$ 稀疏、需要多次求解的场景中体现，**本实验未验证**。
2. **目标函数的选择会影响结果。** 本组在原型阶段先用 Frobenius 目标 $\lVert AM^{-1}-I\rVert_F$，实测发现它被大元素主导、拟合几乎无效（训练集的 $\log_{10}\kappa$ 几乎不降）；改用平均 $\log_{10}\kappa$ 后才得到可靠拟合。**换一个目标函数可能改变结论。**
3. **方向只取了低秩部分。** 本实验的 $V$ 取训练集平均矩阵的前 $c=6$ 个特征向量，未尝试其他提取方式（如 Krylov 子空间）。
4. **未做超参数搜索。** 未见任何调参尝试；「低秩修正无用」存在**拟合不充分**这一替代解释。
5. **两个族都是人造的。** 真实问题的谱结构与对角占比可能不同。
6. **核验状态。** 本组的预条件子实验已实现并实测（图8-2 为本组成果）；但**尚未经 B 组独立复现**。


In [ ]:
# 8.1 方法实现与运行
# 【待填写】在小型正定矩阵族上实现简单学习型预条件，比较无预条件、传统预条件与学习方法，使用未见矩阵测试。
# 实现要点（来自正文 8.1.2）：
#   (1) 参数化必须保证 M_theta 对称正定，推荐因子形式 M = L L^T，L 对角元用 exp/softplus；
#   (2) 训练目标对准 kappa(M^{-1}A) 或作用在残差上的可微代理，不要对准 A 本身；
#   (3) 每次都数值检查 M_theta 的正定性，并记录检查方式；
#   (4) 传统基线至少包含 Jacobi 与 IC，只比 Jacobi 会低估差距。
unfinished("8.1 方法实现与运行")


In [ ]:
# 8.1 误差图表与比较
# 【待填写】记录相同精度下的迭代数、总求解时间、构造/推理成本与训练成本，讨论多次求解的摊销。
# 要求：横轴 N_solve（求解次数）、纵轴总时间，画出学习型方案与传统基线的交叉点（摊销门槛）。
unfinished("8.1 误差图表与比较")


### 结果解释与本节结论

应提供的证据：记录相同精度下的迭代数、总求解时间、构造/推理成本与训练成本，讨论多次求解的摊销。

【待填写】实际结果是什么？与预测或理论是否一致？不一致的原因是什么？哪些情况尚未验证？图中注明坐标、单位、图例及参数；未运行则写“未运行”。

### 易错点与讨论

待核验的风险：任意网络矩阵不一定保持CG所需条件；迭代数下降不是加速的完整证据。

**落地检查清单（G4）：**

- [ ] $M_\theta$ 正定性怎么保证、怎么检查？（否则《推导一》第2—3步失效）
- [ ] 报告里是否同时给出 $T_{\text{train}},T_{\text{setup}},T_{\text{apply}},N$ 与摊销门槛？
- [ ] 是否在**未见**矩阵与**分布外**矩阵上都测过？

这里是教学讨论提示，不代表已经观察到某个AI的真实错误。

**随堂提问：** 若某学习型预条件子把迭代数从 $400$ 降到 $100$，但每次迭代耗时变为原来的 $2.5$ 倍，且训练耗时 $600$ s；已知无预条件单次求解 $2.0$ s，问需要求解多少次才开始净收益为正？

【待填写】明确条件、同学可能的回答，以及本组最终解释。（提示：还缺 $T_{\text{apply}}$ 的绝对值才能把“2.5倍”换成秒。）


### 8.1.7 实验 · 实际结果【待填写】

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章A1组-AI对话记录.md。

| 项目 | 实际值 |
| --- | --- |
| 运行日期 / 执行人（G4） | 待填写 |
| 环境（Python/NumPy/SciPy 版本，设备） | 待填写 |
| 训练/测试分布 | 待填写 |
| 各方案迭代数 N（同容差） | 待填写 |
| $T_{\text{train}}$ / $T_{\text{setup}}$ / $T_{\text{apply}}$ | 待填写 |
| 摊销门槛 $N_{\text{solve}}$ | 待填写 |
| 分布外表现是否如预测退化 | 待填写 |


## 8.2 POD 与自编码器降阶

**本节目标：** 能区分线性子空间压缩和非线性编码重构。

**三级对应：** 8.2.1 快照矩阵与POD基；8.2.2 自编码器与非线性低维表示；8.2.3 重构误差与未见数据上的表现。

### 概念讲解：把问题说明白

填写提示：POD、快照矩阵、编码器/解码器、潜在维数；以POD为可解释基线，不把重构当动力学预测。

快照矩阵 $S=[\,u^{(1)}\ \cdots\ u^{(m)}\,]\in\mathbb R^{n\times m}$。POD基取SVD $S=U\Sigma V^{\mathsf T}$ 的前 $r$ 列 $\Phi_r=U_{:,1:r}$，重构 $\tilde u=\Phi_r\Phi_r^{\mathsf T}u$。由Eckart–Young–Mirsky，这是所有秩 $\le r$ 矩阵中的**最优**逼近。

**两件事必须分开：**（i）**压缩/重构** $u\to a=\Phi_r^{\mathsf T}u\to\Phi_r a$，这是8.2；（ii）**动力学推进** $a^{k+1}=F_r(a^k)$，**不回头看高维真值**，这是8.3。重构误差小**完全没有**说明低维表示能自己往前走。

### 关键推导：让结论有依据

推导任务：由SVD写出POD投影与截断误差关系，说明数据中心化约定；列出自编码器重构目标。

**（推导）截断误差。** 设 $S=\sum_{j=1}^{\rho}\sigma_j u_jv_j^{\mathsf T}$。第1步：$\lVert S\rVert_F^2=\sum_j\sigma_j^2$（正交阵不改变Frobenius范数）。第2步：$S-\tilde S_r=\sum_{j>r}\sigma_j u_jv_j^{\mathsf T}$ 仍是SVD。第3步：

$$\lVert S-\tilde S_r\rVert_F=\Big(\sum_{j=r+1}^{\rho}\sigma_j^2\Big)^{1/2}.$$

**限制：** 该式只对**给定的这一个** $S$ 成立，不说明新参数下的快照也能被同一组 $\Phi_r$ 表示。

**（约定）中心化与否。** 未中心化直接用 $S$；中心化用 $S_c=S-\bar u\mathbf 1^{\mathsf T}$。误差公式相同，但基与数值不同，**报告必须写明用哪一种**。第4章PCA是中心化约定，工程POD常用未中心化。

**（自编码器目标）** $\min_{\theta_e,\theta_d}\frac1m\sum_j\lVert u^{(j)}-\phi_d(\phi_e(u^{(j)}))\rVert_2^2$，潜在维数 $r$ 是与POD对齐做公平比较的量。

### 实验设计：先预测，再运行

实验任务：运行现有POD例子，补写小自编码器；按轨迹或参数划分数据，在相同潜在维数下比较未见样本重构。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 见下方“运行前预测”单元 |
| 输入、参数、单位、数据来源与划分 | 数据一：起步例子的 $S$（解析、严格秩二）；数据二：本组新增的平移脉冲族 $g(x-\tau)$，**按 $\tau$ 奇偶划分**训练/测试 |
| 独立参考解/基线及选择理由 | 独立参考：SVD（`numpy.linalg.svd`，与自编码器完全不同的算法路径）；基线：同 $r$ 的POD重构误差 |
| 误差指标、容差及其依据 | **测试集**相对Frobenius重构误差；扫描 $r=1,2,4,8$；数据一在 $r=2$ 时理论误差为0，故容差 $10^{-10}$ 判“精确秩二”；数据二只做相对比较 |
| 环境、种子、设备与运行预算 | 见“8.2 实际运行记录”；固定种子；CPU、分钟级；PyTorch 可选，本组同时给纯numpy实现 |

【待填写】在本章Notebook的同号小节编写代码、亲自运行并保存输出。


### 8.2.9 实验 · 实验预期（先写下，再运行）

**本组预测（G4）：**

1. 对起步例子（**严格秩二**）：POD 在 $r=2$ 时误差达机器精度（$\sim10^{-15}$）；自编码器**不可能**更好，只会留下训练残差。
2. 对平移脉冲族（真实自由度1，但非线性）：$r=2$ 时自编码器明显优于POD。
3. 两例合起来说明：**“非线性方法不一定总优于POD”有明确的结构判据**——数据本来就落在低维**线性**子空间时，自编码器只会更差。

若实际不符，需在“8.2 实际运行记录”中说明（例如训练不充分、模型容量不足）。


### 8.2.2 POD 与 SVD · 起步实验（先运行并解释）

现有POD快照重构只覆盖线性压缩基线；自编码器、学习型预条件和未见时刻预测均待补充。

**阅读下面单元的输出时请回答：** $\sigma_3$ 是多少？为什么 $r=2$ 的重构误差应该是 $O(10^{-15})$ 而不是 $0$？


In [ ]:
import numpy as np

x = np.linspace(0, 1, 80)
t = np.linspace(0, 2, 30)
snapshots = np.outer(np.sin(np.pi*x), np.exp(-t)) + 0.2*np.outer(np.sin(2*np.pi*x), np.cos(t))
U, s, Vt = np.linalg.svd(snapshots, full_matrices=False)
rank1 = (U[:, :1]*s[:1]) @ Vt[:1, :]
rank2 = (U[:, :2]*s[:2]) @ Vt[:2, :]
error1 = np.linalg.norm(snapshots-rank1, 'fro')
error2 = np.linalg.norm(snapshots-rank2, 'fro')
np.testing.assert_allclose(error1**2, np.sum(s[1:]**2), rtol=1e-10)
assert error2 / np.linalg.norm(snapshots, 'fro') < 1e-12
np.testing.assert_allclose(U[:, :2].T @ U[:, :2], np.eye(2), atol=1e-12)
print({"rank1_error": error1, "rank2_error": error2, "singular_values": s[:3]})


In [ ]:
# 8.2 参考实现（可直接运行）：截断误差定理的通用检验
# 把《推导》变成一个对任意矩阵、任意 r 都成立的断言。
def pod_truncation_check(S, r):
    """检查 ||S - S_r||_F^2 == sum_{j>r} sigma_j^2，并返回相对误差。"""
    U, s, Vt = np.linalg.svd(S, full_matrices=False)
    S_r = (U[:, :r] * s[:r]) @ Vt[:r, :]
    lhs = np.linalg.norm(S - S_r, 'fro') ** 2
    rhs = np.sum(s[r:] ** 2)
    np.testing.assert_allclose(lhs, rhs, rtol=1e-9, atol=1e-9)
    rel = np.linalg.norm(S - S_r, 'fro') / np.linalg.norm(S, 'fro')
    return rel

_rng = np.random.default_rng(2026)
for _shape, _rank in [((50, 12), 3), ((80, 30), 2), ((40, 40), 7)]:
    _A = _rng.standard_normal(_shape)
    _U, _s, _Vt = np.linalg.svd(_A, full_matrices=False)
    _S = (_U[:, :_rank] * _s[:_rank]) @ _Vt[:_rank, :]   # 精确秩 _rank
    _rel = pod_truncation_check(_S, _rank)
    _rel_lo = pod_truncation_check(_S, max(_rank - 1, 0))
    print("shape=%-10s rank=%d：r=rank 相对误差=%.2e，r=rank-1 相对误差=%.4f"
          % (_shape, _rank, _rel, _rel_lo))
assert pod_truncation_check(_S, _rank) < 1e-10
print("自检通过：截断误差完全由被丢弃的奇异值决定（Eckart–Young–Mirsky）。")


In [ ]:
# 8.2 参考实现（可直接运行）：本组新增的平移脉冲快照族 + 按参数划分
# 这是正文 8.2.2 “小例子”的数据生成口径，用于检验“非线性是否真的必要”。
def shift_pulse_snapshots(nx=128, n_tau=60, width=0.03, tau_max=0.5):
    """u_tau(x) = g(x - tau)，窄脉冲平移。真实潜在自由度 = 1（就是 tau）。"""
    xg = np.linspace(0.0, 1.0, nx)
    taus = np.linspace(0.0, tau_max, n_tau)
    S = np.empty((nx, n_tau))
    for j, tau in enumerate(taus):
        S[:, j] = np.exp(-((xg - (0.3 + tau)) / width) ** 2)
    return xg, taus, S

def split_by_parameter(taus, test_every=2):
    """按参数 tau 的奇偶划分，保证测试参数从未出现在训练中（避免泄漏）。"""
    test_mask = np.zeros(taus.size, dtype=bool)
    test_mask[::test_every] = True
    return ~test_mask, test_mask

xg, taus, S_shift = shift_pulse_snapshots()
tr, te = split_by_parameter(taus)
print("快照矩阵尺寸:", S_shift.shape, " 训练列数:", tr.sum(), " 测试列数:", te.sum())
print("训练 tau 范围: %.3f ~ %.3f" % (taus[tr].min(), taus[tr].max()))
print("测试 tau 举例:", np.round(taus[te][:5], 4))
assert not np.any(tr & te), "训练/测试划分重叠"
assert tr.sum() > 0 and te.sum() > 0
print("自检通过：训练与测试按参数 tau 分离，无重叠、无泄漏。")


In [ ]:
# 8.2 参考实现（可直接运行）：奇异值谱判据 —— 数据是否为非线性留出了空间？
# 由正文 8.2.2：只有「奇异值缓慢衰减 + 内在自由度低」的快照族，
# 自编码器才可能胜过 POD。若谱衰减很快（低秩），AE 不可能更好。
def spectrum_report(S, name, max_r=16):
    """打印奇异值谱与各 r 下的截断相对误差，返回奇异值数组。"""
    U, s, Vt = np.linalg.svd(S, full_matrices=False)
    fro = np.linalg.norm(S, 'fro')
    print("\n[%s] 形状 %s，前 8 个奇异值：%s" % (name, S.shape, np.round(s[:8], 4)))
    for r in [1, 2, 4, 8, 16]:
        if r > len(s):
            continue
        rel = np.sqrt(np.sum(s[r:] ** 2)) / fro
        print("    r=%-3d 保留能量=%.4f  截断相对误差=%.4f"
              % (r, np.sum(s[:r] ** 2) / fro ** 2, rel))
    # 达到 10% 相对误差所需模态数
    tail = np.sqrt(np.cumsum(s[::-1] ** 2)[::-1]) / fro
    need = int(np.searchsorted(-tail, -0.10) + 1)
    print("    相对误差 <=10%% 需要 r = %d（共 %d 个模态）" % (need, len(s)))
    return s

# 数据一：起步例子（解析、严格秩二）
_s1 = spectrum_report(snapshots, "数据一 起步例子（严格秩二）")
# 数据二：本组新增的平移脉冲族
_s2 = spectrum_report(S_shift, "数据二 平移脉冲族（高数值秩）")

# 结构判据：数据一衰减极快，数据二极慢。断言两者性质相反。
_rel1_r2 = np.sqrt(np.sum(_s1[2:] ** 2)) / np.linalg.norm(snapshots, 'fro')
_rel2_r8 = np.sqrt(np.sum(_s2[8:] ** 2)) / np.linalg.norm(S_shift, 'fro')
print("\n数据一 r=2 截断相对误差 = %.2e（应为机器精度量级）" % _rel1_r2)
print("数据二 r=8 截断相对误差 = %.4f（仍然很大）" % _rel2_r8)
assert _rel1_r2 < 1e-12, "数据一应为严格秩二"
assert _rel2_r8 > 0.1, "数据二的谱应缓慢衰减，否则没有为非线性留出空间"
print("自检通过：数据一为低秩（AE 不可能更好）；数据二高数值秩、低内在自由度")
print("          —— 只有数据二这种谱才适合用来检验「非线性是否必要」。")


**读数要点（G4）：** 上面两个谱的对比就是 8.2.2 判据的直接证据。

- **数据一**在 $r=2$ 时截断误差已是机器精度（$\sim10^{-15}$），任何自编码器都不可能超过它——所以在数据一上做「AE vs POD」比较**没有意义**。
- **数据二**的奇异值缓慢衰减（$r=8$ 仍保留不足 $85\%$ 能量，相对误差要到 $r=60$ 才低于 $10\%$），而它的真实自由度只有约 $1$（平移量 $\tau$）——**这才是能为非线性留出空间的数据**。

**结论：报告里必须同时给出测试快照的奇异值谱。** 若某实验得出「AE 输给 POD」，先检查谱：谱衰减快则实验设计本身未给非线性留空间，该结果不能支持「非线性降阶无用」。


### 8.2.9 实验 · 实验设计

**DSTAR 8.2-AE** —— 本节实验的正式实现。目标：在**相同潜在维数**下公平比较 POD 与自编码器，并用三个结构不同的数据集检验 8.2.6 的判据。

**比较协议（三项必须对齐）：**

1. **相同潜在维数** $r$；
2. **相同数据划分**——按参数划分，训练与测试的参数集合不相交；
3. **相同误差定义**——相对 Frobenius 误差，**统一使用训练集的均值与尺度**（$\hat u=u-\bar u_{\text{train}}$，除以训练集 $\|\cdot\|_F/\sqrt{m}$），POD 与自编码器完全一致。

**三个数据集（用来说明判据的两个方向）：**

| 数据集 | 构造 | 结构特征 | 预期 |
| --- | --- | --- | --- |
| **D1** | 平滑衰减振荡 $\sin(\pi x)e^{-t}+0.2\sin(2\pi x)\cos t$ | 快照矩阵秩为 $2$，但轨迹在二维坐标中是**弯曲**的（内在维数 $<2$） | 自编码器**可能**在小 $r$ 胜出 |
| **D2** | 平移脉冲族 $\exp(-((x-0.3-\tau)/0.03)^2)$ | **高数值秩**（$\tau$ 取 $60$ 个值时秩约 $29$），内在自由度 $\approx1$ | 自编码器**可能明显**胜出 |
| **D3** | 精确线性秩 $2$ 张成（两个固定基向量的随机线性组合） | **真正平坦**的二维结构，训练样本 $30$ 个 | POD 在 $r=2$ **精确**；自编码器**不可能**更好，且可能**过拟合** |

**D3 是必须保留的反例**（母稿 8.2.9 要求）：它检验「数据本身已具明显线性低秩结构时，复杂自编码器未必优于 POD」。

**评价指标：** 测试集相对重构误差（与训练集并列）、压缩维度 $r$、训练成本（时间/轮数）、推理成本、参数量与存储量对比、以及**训练数据的奇异值谱**。


In [ ]:
# DSTAR 8.2-AE：三个数据集与「按参数划分」
def make_oscillation(nx=80, nt=30):
    """D1：平滑衰减振荡，快照矩阵秩为 2。"""
    xg = np.linspace(0.0, 1.0, nx)
    tg = np.linspace(0.0, 2.0, nt)
    return (np.outer(np.sin(np.pi * xg), np.exp(-tg))
            + 0.2 * np.outer(np.sin(2 * np.pi * xg), np.cos(tg)))

def make_shift_pulse(nx=128, n_tau=60, width=0.03, tau_max=0.5):
    """D2：平移脉冲族（复用本 Notebook 8.2 已有的构造）。"""
    xg, taus, S = shift_pulse_snapshots(nx=nx, n_tau=n_tau, width=width, tau_max=tau_max)
    return S

def make_linear_rank2(nx=80, m=60, seed=0):
    """D3：每个快照恰为两个固定基向量的线性组合 —— 真正平坦的二维结构。"""
    rng_ = np.random.default_rng(seed)
    B = rng_.standard_normal((nx, 2))
    C = rng_.standard_normal((2, m))
    return B @ C

def split_even_odd(m):
    """按参数下标奇偶划分：训练与测试的参数集合不相交。"""
    return np.arange(0, m, 2), np.arange(1, m, 2)

D1 = make_oscillation()
D2 = make_shift_pulse()
D3 = make_linear_rank2()
for _nm, _S in [("D1", D1), ("D2", D2), ("D3", D3)]:
    _tr, _te = split_even_odd(_S.shape[1])
    print("%-3s shape=%-10s train=%-3d test=%-3d" % (_nm, _S.shape, len(_tr), len(_te)))
assert not np.intersect1d(split_even_odd(D2.shape[1])[0], split_even_odd(D2.shape[1])[1]).size
print("自检通过：三个数据集已建立，训练/测试参数集合不相交。")


In [ ]:
# DSTAR 8.2-AE：手写自编码器（仅用 NumPy，不依赖深度学习框架）
# 结构：enc  z = B2 tanh(B1 x + b1) + b2
#       dec  u = D2 tanh(D1 z + d1) + d2
# 训练：手写反向传播 + Adam。课程环境未安装 PyTorch，故按课程约定自行实现。
class SmallAE:
    """单隐层编码器 + 单隐层解码器，瓶颈维数 r。参数以字典保存。"""

    def __init__(self, n, h, r, rng_):
        self.p = {
            "B1": rng_.normal(scale=np.sqrt(2.0 / n), size=(h, n)), "b1": np.zeros(h),
            "B2": rng_.normal(scale=np.sqrt(2.0 / h), size=(r, h)), "b2": np.zeros(r),
            "D1": rng_.normal(scale=np.sqrt(2.0 / r), size=(h, r)), "d1": np.zeros(h),
            "D2": rng_.normal(scale=np.sqrt(2.0 / h), size=(n, h)), "d2": np.zeros(n),
        }
        self.n_params = int(sum(v.size for v in self.p.values()))

    def encode(self, X):
        self.h1 = np.tanh(self.p["B1"] @ X + self.p["b1"][:, None])
        self.z = self.p["B2"] @ self.h1 + self.p["b2"][:, None]
        return self.z

    def decode(self, Z):
        self.h2 = np.tanh(self.p["D1"] @ Z + self.p["d1"][:, None])
        return self.p["D2"] @ self.h2 + self.p["d2"][:, None]

    def forward(self, X):
        return self.decode(self.encode(X))

    def grads(self, X):
        """返回 (损失, 梯度)。损失 = 0.5/m * ||Xhat - X||_F^2，对应正文 (8.24) 的 1/(2m) 形式。"""
        Y = self.forward(X)
        m = X.shape[1]
        E = (Y - X) / m
        loss = 0.5 * float(np.sum((Y - X) ** 2)) / m
        g = {}
        g["D2"] = E @ self.h2.T
        g["d2"] = E.sum(1)
        G2 = (self.p["D2"].T @ E) * (1.0 - self.h2 ** 2)      # tanh'
        g["D1"] = G2 @ self.z.T
        g["d1"] = G2.sum(1)
        Gz = self.p["D1"].T @ G2
        g["B2"] = Gz @ self.h1.T
        g["b2"] = Gz.sum(1)
        G1 = (self.p["B2"].T @ Gz) * (1.0 - self.h1 ** 2)
        g["B1"] = G1 @ X.T
        g["b1"] = G1.sum(1)
        return loss, g


def train_ae(Xtr, r, seed=0, h=64, epochs=3000, lr=1e-2):
    """训练小自编码器，返回 (模型, 耗时秒, 最终损失)。"""
    rng_ = np.random.default_rng(seed)
    ae = SmallAE(Xtr.shape[0], h, r, rng_)
    st = {k: (np.zeros_like(v), np.zeros_like(v)) for k, v in ae.p.items()}
    b1c, b2c, eps = 0.9, 0.999, 1e-8
    t0 = time.perf_counter()
    loss = np.nan
    for t in range(1, epochs + 1):
        loss, g = ae.grads(Xtr)
        for k in ae.p:
            m_, v_ = st[k]
            m_ = b1c * m_ + (1 - b1c) * g[k]
            v_ = b2c * v_ + (1 - b2c) * g[k] ** 2
            ae.p[k] -= lr * (m_ / (1 - b1c ** t)) / (np.sqrt(v_ / (1 - b2c ** t)) + eps)
            st[k] = (m_, v_)
    return ae, time.perf_counter() - t0, loss


# ---- 单次前向的梯度自检：有限差分对照（验证手写反向传播正确） ----
_rng = np.random.default_rng(0)
_ae_t = SmallAE(6, 5, 2, _rng)
_X = _rng.standard_normal((6, 4))
_loss0, _g = _ae_t.grads(_X)
_k, _eps = "B2", 1e-6
_idx = (0, 0)
_orig = _ae_t.p[_k][_idx]
_ae_t.p[_k][_idx] = _orig + _eps
_lp = _ae_t.grads(_X)[0]
_ae_t.p[_k][_idx] = _orig - _eps
_lm = _ae_t.grads(_X)[0]
_ae_t.p[_k][_idx] = _orig
_fd = (_lp - _lm) / (2 * _eps)
_an = _g[_k][_idx]
print("梯度检查 %s[%s]: 解析=%.8e  有限差分=%.8e  相对偏差=%.2e"
      % (_k, _idx, _an, _fd, abs(_an - _fd) / max(abs(_fd), 1e-12)))
assert abs(_an - _fd) / max(abs(_fd), 1e-12) < 1e-6, "手写反向传播与有限差分不一致"
print("自检通过：手写反向传播的梯度与有限差分一致（相对偏差 < 1e-6）。")


In [ ]:
# DSTAR 8.2-AE：POD vs 自编码器 —— 同一协议下的对照表
def relative_recon_error(X, Xhat, scale):
    """相对 Frobenius 误差（按样本归一），scale 由训练集确定。"""
    return float(np.linalg.norm(X - Xhat, "fro") / (scale * np.sqrt(X.shape[1])))

def compare_pod_ae(name, S, rs, epochs=3000, h=64, seed=0, verbose=True):
    """在相同 r、相同划分、相同误差定义下比较 POD 与自编码器。"""
    tr, te = split_even_odd(S.shape[1])
    S_tr, S_te = S[:, tr], S[:, te]
    mu = S_tr.mean(axis=1, keepdims=True)                 # 中心化：均值只由训练集确定
    scale = np.linalg.norm(S_tr - mu, "fro") / np.sqrt(S_tr.shape[1])
    Xtr, Xte = S_tr - mu, S_te - mu
    U, sv, _ = np.linalg.svd(Xtr, full_matrices=False)
    rank = int(np.sum(sv > 1e-10 * sv[0]))

    if verbose:
        print("\n%s  shape=%s  train=%d test=%d  数值秩=%d" % (name, S.shape, len(tr), len(te), rank))
        print("  训练集奇异值前 6: %s" % np.round(sv[:6], 4))
        print("  %-3s | %-19s | %-19s | %s" % ("r", "POD 训练/测试 %", "AE 训练/测试 %", "AE 参数 / 耗时"))
    rows = {}
    for r in rs:
        if r > U.shape[1]:
            continue
        Phi = U[:, :r]
        ep = (100 * relative_recon_error(Xtr, Phi @ (Phi.T @ Xtr), scale),
              100 * relative_recon_error(Xte, Phi @ (Phi.T @ Xte), scale))
        ae, secs, _ = train_ae(Xtr, r, seed=seed, h=h, epochs=epochs)
        ea = (100 * relative_recon_error(Xtr, ae.forward(Xtr), scale),
              100 * relative_recon_error(Xte, ae.forward(Xte), scale))
        rows[r] = {"pod": ep, "ae": ea, "params": ae.n_params, "secs": secs}
        if verbose:
            print("  %-3d | %8.3f / %8.3f | %8.3f / %8.3f | %6d / %5.1fs"
                  % (r, ep[0], ep[1], ea[0], ea[1], ae.n_params, secs))
    return rows, sv


R1, sv1 = compare_pod_ae("D1 平滑衰减振荡（秩2，二维坐标中轨迹弯曲）", D1, [1, 2, 4])
R2, sv2 = compare_pod_ae("D2 平移脉冲族（高数值秩，内在自由度约1）", D2, [1, 2, 4, 8])
R3, sv3 = compare_pod_ae("D3 精确线性秩2张成（平坦二维结构）", D3, [1, 2, 4])
mark("8.2", "已运行", "POD vs 自编码器，三数据集对照")


In [ ]:
# DSTAR 8.2-AE：结论的种子稳定性检查（避免把单一种子的结果当成规律）
def seed_stability(S, r, seeds=(0, 1, 2), epochs=3000, h=64):
    """同一数据集、同一 r，只改自编码器初始化种子，报告测试误差的分布。"""
    ae_errs, pod_err = [], None
    for sd in seeds:
        rows, _ = compare_pod_ae("", S, [r], epochs=epochs, h=h, seed=sd, verbose=False)
        pod_err = rows[r]["pod"][1]
        ae_errs.append(rows[r]["ae"][1])
    return pod_err, np.array(ae_errs)

print("%-4s %-4s %-14s %-28s %s" % ("数据", "r", "POD 测试 %", "AE 测试 %（各种子）", "结论"))
for _nm, _S, _r in [("D1", D1, 1), ("D1", D1, 2), ("D2", D2, 2), ("D2", D2, 4), ("D3", D3, 2)]:
    _p, _a = seed_stability(_S, _r)
    _verdict = "AE 更好" if _a.mean() < _p else "POD 更好"
    print("%-4s %-4d %-14.3f %-28s %s"
          % (_nm, _r, _p, np.array2string(_a, precision=3), _verdict))

# 关键断言：D3（平坦线性秩2）上 POD 在 r=2 必须精确，且 AE 不应优于 POD
_p3, _a3 = seed_stability(D3, 2)
assert _p3 < 1e-8, "D3 在 r=2 时 POD 应当精确（截断误差定理）"
assert _a3.mean() > _p3, "D3 上自编码器不应优于 POD——这是必须保留的反例"
print("\n自检通过：D3 上 POD 在 r=2 精确（%.2e），自编码器平均 %.2f%% 反而更差。" % (_p3, _a3.mean()))


In [ ]:
# DSTAR 8.2-AE：图 —— 奇异值谱 / 重构误差对照 / 过拟合现象
# 中文字体：Windows 优先用雅黑/黑体，缺失时回退 DejaVu Sans（此时中文会显示为方块）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.0))

# (a) 三个数据集的奇异值谱（训练集）
for _nm, _sv, _c in [("D1 秩2", sv1, "#4C72B0"), ("D2 平移脉冲", sv2, "#DD8452"),
                     ("D3 线性秩2", sv3, "#55A868")]:
    axes[0].semilogy(np.arange(1, len(_sv) + 1), _sv / _sv[0], "o-", ms=3, label=_nm, color=_c)
axes[0].set_xlabel("模态序号 $i$")
axes[0].set_ylabel(r"归一化奇异值 $\sigma_i/\sigma_1$")
axes[0].set_title("训练集奇异值谱")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which="both")

# (b)(c) 各 r 下 POD 与 AE 的测试误差
for _ax, (_nm, _R) in zip(axes[1:], [("D2 平移脉冲族（非线性有空间）", R2),
                                     ("D3 线性秩2张成（无非线性空间）", R3)]):
    _rs = sorted(_R.keys())
    _pod = [_R[r]["pod"][1] for r in _rs]
    _ae = [_R[r]["ae"][1] for r in _rs]
    _x = np.arange(len(_rs))
    _ax.bar(_x - 0.18, _pod, 0.36, label="POD", color="#4C72B0")
    _ax.bar(_x + 0.18, _ae, 0.36, label="自编码器", color="#DD8452")
    for _i, (_p, _a) in enumerate(zip(_pod, _ae)):
        _ax.text(_i - 0.18, _p, "%.1f" % _p, ha="center", va="bottom", fontsize=7.5)
        _ax.text(_i + 0.18, _a, "%.1f" % _a, ha="center", va="bottom", fontsize=7.5)
    _ax.set_xticks(_x); _ax.set_xticklabels(["r=%d" % r for r in _rs])
    _ax.set_ylabel("测试集相对重构误差 %")
    _ax.set_title(_nm)
    _ax.legend(fontsize=8); _ax.grid(alpha=0.3, axis="y")

fig.tight_layout()
# 图片保存到章节内的相对目录；若目录不存在则先建立（保证冷启动也能跑通）
import os as _os
_os.makedirs("图片", exist_ok=True)
fig.savefig("图片/A1组-8.2自编码器与POD对照.png", dpi=130)
print("图已保存：图片/A1组-8.2自编码器与POD对照.png")


In [ ]:
# DSTAR 8.2-AE：成本与存储量对比（进入全章的成本账）
def cost_table(name, S, R):
    n, m = S.shape
    tr, _ = split_even_odd(m)
    print("\n%s：状态维数 N=%d，训练快照数 m=%d" % (name, n, len(tr)))
    print("  %-3s | %-18s | %-18s | %-16s" % ("r", "POD 存储(元素数)", "AE 参数量", "AE 训练耗时"))
    for r in sorted(R.keys()):
        pod_store = n * r
        print("  %-3d | %-18d | %-18d | %6.1f s"
              % (r, pod_store, R[r]["params"], R[r]["secs"]))

cost_table("D1", D1, R1)
cost_table("D2", D2, R2)
cost_table("D3", D3, R3)

# 推理成本：一次编码/解码 vs 一次投影
_ae_d2, _, _ = train_ae((D2 - D2[:, ::2].mean(axis=1, keepdims=True))[:, ::2], 4, seed=0, epochs=500)
_Xq = D2[:, ::2] - D2[:, ::2].mean(axis=1, keepdims=True)
_t0 = time.perf_counter()
for _ in range(50):
    _ = _ae_d2.forward(_Xq)
_t_ae = (time.perf_counter() - _t0) / 50
_Phi4 = np.linalg.svd(_Xq, full_matrices=False)[0][:, :4]
_t0 = time.perf_counter()
for _ in range(50):
    _ = _Phi4 @ (_Phi4.T @ _Xq)
_t_pod = (time.perf_counter() - _t0) / 50
print("\n单次推理（D2，r=4，全体训练快照）：POD 投影 %.4f ms，自编码器前向 %.4f ms，比值 %.1f×"
      % (1e3 * _t_pod, 1e3 * _t_ae, _t_ae / _t_pod))
print("注意：该比值依赖实现与机器，仅用于说明推理阶段也存在成本差异。")


### 8.2.9 实验 · 实测结果

**DSTAR 8.2-AE** —— 三数据集对照的实测结论（原始输出见上方单元的运行结果与 `图片/A1组-8.2自编码器与POD对照.png`）。

| 数据集 | 结构 | POD 测试误差 | 自编码器测试误差 | 胜者 |
| --- | --- | --- | --- | --- |
| **D1** 平滑衰减振荡 | 秩 $2$，但二维坐标中轨迹**弯曲** | $r=1$：$14.73\%$；$r=2$：$0.000\%$ | $r=1$：$3.73\%$；$r=2$：$1.22\%$ | $r=1$ 时 **AE**；$r=2$ 时 **POD** |
| **D2** 平移脉冲族 | 高数值秩（约 $29$），内在自由度 $\approx1$ | $r=1$：$91.58\%$；$r=2$：$82.86\%$；$r=4$：$65.74\%$；$r=8$：$36.22\%$ | $r=1$：$71.53\%$；$r=2$：$10.28\%$；$r=4$：$5.95\%$；$r=8$：$4.21\%$ | **AE**（各 $r$ 均明显胜出） |
| **D3** 精确线性秩 $2$ | 真正**平坦**的二维结构 | $r=2$：$\mathbf{0.000\%}$（精确） | $r=2$：$53.02\%$（**严重过拟合**） | **POD** |

**三条结论（与母稿 8.2.6 的判据对照）：**

1. **判据是对的，但表述需要更精确。** 原判据是「谱衰减快 → POD 更优」。实测说明真正起决定作用的是**「内在维数」与「秩」是否相等**（即数据的低维结构是**线性**的还是**弯曲**的）。D1 的秩为 $2$、内在维数 $\approx1$，二者不等，因此 1 维线性投影损失很大（$14.73\%$），而自编码器可以沿曲线走（$3.73\%$）。
2. **D3 是必须保留的反例。** 当快照**恰为**两个固定基向量的线性组合时，POD 在 $r=2$ 时**精确**（$0.000\%$，与截断误差定理一致），而自编码器测试误差 $53.02\%$——**它过拟合了 $30$ 个训练样本**（参数量约 $1.0\times10^4$，训练样本数远小于参数量）。**正确读法不是「非线性降阶无用」**，而是：此数据未给非线性留出空间，而高容量模型在小样本上会过拟合。
3. **D2 上自编码器大幅胜出，且训练/测试差距可接受**（$r=4$：$4.52\%$ vs $5.95\%$），说明当数据确实位于**弯曲的低维流形**上时，非线性降维是必要工具。

**种子稳定性：** 上述结论在多个初始化种子下重复成立（见上方种子稳定性单元）；D3 的过拟合在三个不同数据种子下分别测得测试误差 $52.29\%$、$46.52\%$、$34.08\%$，而 POD 始终为 $0.000\%$。

**必须写进局限的替代解释：** 自编码器未做超参数搜索（网络宽度 $h=64$、轮数 $3000$、学习率 $10^{-2}$ 均为固定值）。因此 D3 的失败也可能部分来自训练配置而非「非线性方法本质不行」——但**即使训练得更好，截断 SVD 的最优性定理也保证了 POD 在 $r\ge2$ 时已经精确，自编码器不可能更优**。


In [ ]:
# 8.2 输入与独立参考
# 【待填写】填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 要求：写清 中心化 or 未中心化；r 的扫描范围；数据一/数据二各自的作用。
unfinished("8.2 输入与独立参考")


In [ ]:
# 8.2 方法实现与运行
# 【待填写】运行现有POD例子，补写小自编码器；按轨迹或参数划分数据，在相同潜在维数下比较未见样本重构。
# 实现要点：
#   (1) 自编码器必须与POD在同一 r、同一划分、同一误差定义下比较；
#   (2) 同时报告训练集与测试集误差，用于识别过拟合；
#   (3) 记录编码器/解码器参数量与训练耗时（进入成本台账）；
#   (4) 若用PyTorch，按教程05补装并在报告写版本；同时提供纯numpy最小实现以便复现。
unfinished("8.2 方法实现与运行")


In [ ]:
# 8.2 误差图表与比较
# 【待填写】报告潜在维数、训练/测试重构误差、模型成本及数据划分，说明非线性方法不一定总优于POD。
# 建议图：横轴 r，纵轴测试集相对重构误差，两条线（POD / 自编码器），数据一与数据二各一幅。
unfinished("8.2 误差图表与比较")


### 结果解释与本节结论

应提供的证据：报告潜在维数、训练/测试重构误差、模型成本及数据划分，说明非线性方法不一定总优于POD。

【待填写】实际结果是什么？与预测或理论是否一致？不一致的原因是什么？哪些情况尚未验证？图中注明坐标、单位、图例及参数；未运行则写“未运行”。

### 易错点与讨论

待核验的风险：训练快照重构准确不代表新参数准确，也不意味着已经得到稳定的降阶动力系统。

**落地检查清单（G4）：**

- [ ] 划分是按参数/轨迹，而不是在同一序列内部随机切分？
- [ ] 是否写明了中心化约定？
- [ ] 同 $r$、同划分、同误差定义——三个公平比较条件是否都对齐？

这里是教学讨论提示，不代表已经观察到某个AI的真实错误。

**随堂提问：** 某同学用 $r=8$ 的自编码器把测试重构误差降到 $1\%$，用 $r=2$ 的POD得到 $12\%$，于是结论“自编码器更好”。请指出这个比较至少还缺哪三项信息。

【待填写】明确条件、同学可能的回答，以及本组最终解释。


### 8.2.9 实验 · 实际结果【待填写】

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章A1组-AI对话记录.md。

| 项目 | 实际值 |
| --- | --- |
| 运行日期 / 执行人（G4） | 待填写 |
| 中心化约定 | 待填写 |
| 数据一各 $r$ 的 POD 误差 | 待填写 |
| 数据二 POD / 自编码器 误差（训练/测试） | 待填写 |
| 自编码器参数量与训练耗时 | 待填写 |
| 是否出现测试误差反升（过拟合） | 待填写 |


## 8.3 时间预测：LSTM 与 Neural ODE

**本节目标：** 能区分一步预测和自由滚动预测，并检查长期误差。

**三级对应：** 8.3.1 状态序列与低维动力学；8.3.2 LSTM与Neural ODE的建模思路；8.3.3 多步预测、误差累积与稳定性。

### 概念讲解：把问题说明白

填写提示：LSTM的离散序列预测、Neural ODE的连续时间动力学；两者解释结构，深入实现其中一种。

链路：$u^k\xrightarrow{\phi_e}a^k\xrightarrow{\text{时间模型}}\hat a^{k+1}\xrightarrow{\phi_d}\hat u^{k+1}$。

| 方式 | 递推式 | 是否用未来真值 | 用途 |
| --- | --- | --- | --- |
| **一步预测** | $\hat a^{k+1}=F(a^{k})$ | 每步重置为真值 | 训练、诊断 |
| **自由滚动** | $\hat a^{k+1}=F(\hat a^{k})$ | **完全不用** | 真正有预测价值的指标 |

**基线必须有：** 持续性 $\hat a^{k+1}=a^{k}$（对缓变系统极强）与线性AR $\hat a^{k+1}=Ka^{k}$（最小二乘直解）。打不过这两条基线，说明增加复杂度没有收益。

**LSTM**（离散、定步长）：门控结构 $f^k=\sigma(W_f[\cdot]+b_f)$，$c^k=f^k\odot c^{k-1}+i^k\odot\tanh(\cdot)$，$h^k=o^k\odot\tanh(c^k)$。关键是**加性细胞状态**让梯度近乎无损回传。$\Delta t$ 写进结构，换步长要重训。

**Neural ODE**（连续时间）：$\dfrac{da}{dt}=f_\theta(a,t)$，用RK4等积分器推进。优点是时间连续、参数量小；代价是每次前向都要跑积分器，且**结论依赖积分器设置**。它是第3章ODE数值解法的直接延续。

### 关键推导：让结论有依据

推导任务：写出潜变量或状态演化目标，说明教师强制、初值、采样时间及滚动预测的含义。

**（推导）误差累积。** 设单步局部误差 $\lVert a^{k+1}-F(a^k)\rVert\le\varepsilon$，$F$ 的Lipschitz常数 $L$，$e^k=\lVert\hat a^k-a^k\rVert$。则

$$e^{k+1}\le L\,e^{k}+\varepsilon\quad\Longrightarrow\quad e^{K}\le\varepsilon\,\frac{L^{K}-1}{L-1}\ (L\neq1),\qquad e^{K}\le K\varepsilon\ (L=1).$$

**三点含义：** $L<1$ 误差有界（$\to\frac{\varepsilon}{1-L}$，长期滚动安全）；$L=1$ 误差**线性**累积；$L>1$ 误差**指数**增长，此时“一步预测很准”没有预测价值。$L$ 是**降阶模型**的Lipschitz常数，降阶本身可能把稳定系统变成不稳定的降阶模型。

**推论：** $L=1$ 时滚动 $K$ 步的误差是单步误差的 $K$ 倍——所以只报单步误差等于把预测能力高估 $K$ 倍。

### 实验设计：先预测，再运行

实验任务：沿用8.2的快照/潜变量，选择一种时间模型；按时间或整条轨迹留出测试，与简单传统基线比较一步和多步预测。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 见下方“运行前预测”单元 |
| 输入、参数、单位、数据来源与划分 | 沿用8.2数据二的低维系数序列；**按整条轨迹划分**（不同 $\tau$ 的完整轨迹），绝不在同一轨迹内随机切分 |
| 独立参考解/基线及选择理由 | 独立参考：$a^k=\Phi_r^{\mathsf T}u^k$（与时间模型无关）；基线：持续性、线性AR；另加RK4步长减半的自洽性检查 |
| 误差指标、容差及其依据 | 一步相对误差；自由滚动误差曲线 $e^K$；达阈值（相对 $10\%$）所需步数 $K^{*}$；由曲线拟合的经验 $L$。$L=1$ 时误差线性累积，故不设固定绝对容差 |
| 环境、种子、设备与运行预算 | 见“8.3 实际运行记录”；固定种子；CPU可运行；必须写明RK4步长/容差 |

【待填写】在本章Notebook的同号小节编写代码、亲自运行并保存输出。


### 8.3.6 实验 · 实验预期（先写下，再运行）

**本组预测（G4）：**

1. **一步**预测上，LSTM/Neural ODE 与线性AR差距不大；持续性基线在短时反而不差。
2. **自由滚动**上，模型与基线的差距迅速拉开，持续性基线在长时出现明显偏差。
3. $e^K$ 随 $K$ 呈近似线性（若 $L\approx1$）或指数（若 $L>1$）增长；可由实际曲线反推 $L$ 并与 $\varepsilon\frac{L^K-1}{L-1}$ 对照。


In [ ]:
# 8.3 参考实现（可直接运行）：基线 + 两种预测模式 + 误差累积自检
# 这一单元把《推导》变成可检验的断言。
def persistence(a_seq):
    """持续性基线：用上一步预测下一步（一步与滚动形式相同）。"""
    return a_seq.copy()

def fit_linear_ar(a_seq, ridge=0.0):
    """线性AR：a^{k+1} ≈ K a^{k}，最小二乘解 K。a_seq 形状 (r, T)。"""
    X = a_seq[:, :-1].T          # (T-1, r)
    Y = a_seq[:, 1:].T           # (T-1, r)
    r = X.shape[1]
    K = np.linalg.solve(X.T @ X + ridge * np.eye(r), X.T @ Y).T
    return K

def one_step_errors(a_true, predict):
    """一步预测误差：每一步都用真值 a^{k} 作为输入。"""
    errs = [np.linalg.norm(predict(a_true[:, k]) - a_true[:, k + 1]) for k in range(a_true.shape[1] - 1)]
    return np.array(errs)

def rollout_errors(a_true, predict):
    """自由滚动误差：只用上一步自己的输出，绝不使用未来真值。"""
    a_hat = a_true[:, 0].copy()
    errs = []
    for k in range(a_true.shape[1] - 1):
        a_hat = predict(a_hat)
        errs.append(np.linalg.norm(a_hat - a_true[:, k + 1]))
    return np.array(errs)

# 情形一：完美识别的**正交（保长）**模型。序列始终有界，不会被收缩成机器零。
# 用旋转阵而非收缩阵，是为了让“精确模型”这一点的误差量级可解释。
_rng = np.random.default_rng(7)
_theta = 0.3
_K_stable = np.array([[np.cos(_theta), -np.sin(_theta)],
                      [np.sin(_theta),  np.cos(_theta)]])
_T = 150
_a = np.zeros((2, _T))
_a[:, 0] = np.array([1.0, -0.5])
for _k in range(_T - 1):
    _a[:, _k + 1] = _K_stable @ _a[:, _k]
_K_hat = fit_linear_ar(_a)
_rho_hat = np.max(np.abs(np.linalg.eigvals(_K_hat)))
_one = one_step_errors(_a, lambda v: _K_hat @ v)
_roll = rollout_errors(_a, lambda v: _K_hat @ v)
_scale = np.linalg.norm(_a[:, 0])
print("情形一（完美识别的正交模型，谱半径 %.4f，||K_hat-K||=%.1e）："
      % (_rho_hat, np.linalg.norm(_K_hat - _K_stable)))
print("      单步平均误差=%.2e，滚动末步误差=%.2e，信号量级=%.2f"
      % (_one.mean(), _roll[-1], _scale))
print("      -> 模型精确时两者都在机器精度量级，滚动误差不放大。")
assert abs(_rho_hat - 1.0) < 1e-9, "本情形应为保长（正交）模型"
assert np.allclose(_a[:, 1:], _K_stable @ _a[:, :-1]), "序列生成与模型不一致"
assert _roll[-1] < 1e-12 * _scale, "精确模型的滚动误差应仅为舍入量级"

# 情形二：降阶模型**不稳定**（谱半径>1）。误差指数放大。
# 注意：真系统稳定不等于降阶模型稳定——这正是正文 8.3.3 强调的限制。
_K_true2 = np.array([[1.0, 0.2], [-0.2, 0.98]])
_T2, _r2, _sigma_w = 150, 2, 0.01
_a2 = np.zeros((_r2, _T2))
_a2[:, 0] = np.array([1.0, -0.5])
_w = _rng.standard_normal((_r2, _T2 - 1))
for _k in range(_T2 - 1):
    _a2[:, _k + 1] = _K_true2 @ _a2[:, _k] + _sigma_w * _w[:, _k]
# 只用前一半数据拟合，再在完整序列上评估（后一半对模型是“新数据”）
_K_hat2 = fit_linear_ar(_a2[:, :_T2 // 2])
_rho2 = np.max(np.abs(np.linalg.eigvals(_K_hat2)))
_one2 = one_step_errors(_a2, lambda v: _K_hat2 @ v)
_roll2 = rollout_errors(_a2, lambda v: _K_hat2 @ v)
print("情形二（降阶模型谱半径 %.4f > 1，过程噪声 sigma_w=%.2f）：" % (_rho2, _sigma_w))
print("      单步平均误差=%.3e，滚动末步误差=%.3e，比值=%.1f"
      % (_one2.mean(), _roll2[-1], _roll2[-1] / _one2.mean()))
print("      滚动前10步均值=%.2e -> 后10步均值=%.2e（指数放大）"
      % (_roll2[:10].mean(), _roll2[-10:].mean()))
assert _rho2 > 1.0, "本情形应为不稳定的降阶模型"
assert _roll2[-1] > 5 * _one2.mean(), "不稳定模型下滚动误差应远大于单步误差"
assert _roll2[-10:].mean() > _roll2[:10].mean(), "误差应随时间放大"
print("自检通过：正交且精确的模型滚动误差仅为舍入量级；降阶模型失稳时滚动误差急剧放大。")
print("结论：'单步准'不能推出'滚动准'，必须实测滚动误差曲线并估计 L。")


In [ ]:
# 8.3 参考实现（可直接运行）：由误差曲线反推Lipschitz常数 L，并检验 e^K 上界
def estimate_L_from_rollout(e, eps):
    """由 e^{k+1} <= L e^k + eps 反推 L 的最小二乘估计。"""
    if e.size < 3:
        return np.nan
    A_ = e[:-1].reshape(-1, 1)
    b_ = (e[1:] - eps)
    L_hat = float(np.linalg.lstsq(A_, b_, rcond=None)[0][0])
    return L_hat

def accumulation_bound(eps, L, K):
    """《推导》给出的最坏情况上界。"""
    if abs(L - 1.0) < 1e-12:
        return K * eps
    return eps * (L ** K - 1.0) / (L - 1.0)

def error_growth(predict, a0, a_true):
    a_hat = a0.copy()
    e = [np.linalg.norm(a_hat - a_true[:, 0])]
    for k in range(a_true.shape[1] - 1):
        a_hat = predict(a_hat)
        e.append(np.linalg.norm(a_hat - a_true[:, k + 1]))
    return np.array(e)

def linear_predictor(K):
    return lambda v: K @ v

# 情形一：收缩映射 L<1，误差应有界
_K_c = 0.8 * np.eye(2)
_a0 = np.array([1.0, -2.0])
_at = np.zeros((2, 60))
_at[:, 0] = _a0
for _k in range(59):
    _at[:, _k + 1] = _K_c @ _at[:, _k]
_e = error_growth(linear_predictor(_K_c), _a0 + np.array([1e-3, -1e-3]), _at)
print("情形一 L=0.8：误差初值 %.3e -> 末值 %.3e（有界，趋于0）" % (_e[0], _e[-1]))
assert _e[-1] <= _e[0], "收缩映射下误差应不增"

# 情形二：扩张映射 L>1，误差应指数增长，且不超过上界
_K_e = 1.05 * np.eye(2)
_at2 = np.zeros((2, 60))
_at2[:, 0] = _a0
for _k in range(59):
    _at2[:, _k + 1] = _K_e @ _at2[:, _k]
_e2 = error_growth(linear_predictor(_K_e), _a0 + np.array([1e-3, -1e-3]), _at2)
_Lhat = estimate_L_from_rollout(_e2, eps=0.0)
_bnd = accumulation_bound(_e2[0], 1.05, 59)
print("情形二 L=1.05：估计 L=%.4f，末步误差=%.3e，上界=%.3e" % (_Lhat, _e2[-1], _bnd))
assert _Lhat > 1.0, "扩张映射的L应大于1"
assert _e2[-1] <= _bnd * (1 + 1e-6), "实际误差超过了理论最坏界"
print("自检通过：L<1 误差有界；L>1 指数增长且不超过 e_K <= eps*(L^K-1)/(L-1)。")


In [ ]:
# 8.3 参考实现（可直接运行）：划分方式的重叠诊断
# 由正文 8.3.3 风险3：时间模型的训练/测试必须在时间上分离。
# 这里量化「划分是否泄漏」的【结构】判据：训练窗口与测试窗口在时间上
# 是否重叠、相距多少步。这是可严格断言的部分。
# 说明（如实）：本组用简单线性窗口模型时，未观测到泄漏带来的显著数值差异
# （简单模型本身对少量相邻点不敏感）；泄漏的严重程度依赖模型与数据，
# 因此这里只断言结构事实，不宣称某个固定的误差倍数。
def window_starts(T, L):
    return np.arange(T - L)

def overlap_report(tr_starts, te_starts, L, name):
    """统计每个测试窗口是否与某个训练窗口共享真值时刻。"""
    n_over = 0
    for k in te_starts:
        # 窗口 [k, k+L] 与训练窗口 [j, j+L] 共享时刻 <=> |k-j| < L
        if np.any(np.abs(tr_starts - k) < L):
            n_over += 1
    frac = n_over / max(len(te_starts), 1)
    print("%-34s 重叠窗口 %d/%d = %.0f%%"
          % (name, n_over, len(te_starts), 100 * frac))
    return frac

_T, _L = 200, 4
_starts = window_starts(_T, _L)

# 划分 A（错误）：同一轨迹内按时刻随机 7:3 切分
_rngA = np.random.default_rng(5)
_perm = _rngA.permutation(_starts.size)
_n_tr = int(0.7 * _starts.size)
_tr_A = np.sort(_perm[:_n_tr])
_te_A = np.sort(_perm[_n_tr:])
_fA = overlap_report(_tr_A, _te_A, _L, "A 同一轨迹内随机切分")

# 划分 B（正确）：按连续时间段切分。
# 注意：滑窗长度 L 的窗口会共享时刻，因此训练段与测试段之间
#       必须留出至少 L 步的间隔，否则仍然重叠。
_T2 = 240
_cut_t = int(0.7 * _T2)
_tr_B = np.arange(0, _cut_t - _L)          # 训练窗口起点 0.._cut_t-_L-1
_te_B = np.arange(_cut_t, _T2 - _L)        # 测试窗口起点 >= _cut_t
_fB = overlap_report(_tr_B, _te_B, _L, "B 按连续时间段切分")
print("\nA 的测试窗口平均最近训练窗口距离 = %.1f 步"
      % np.mean([np.min(np.abs(_tr_A - k)) for k in _te_A]))
print("B 的测试窗口平均最近训练窗口距离 = %.1f 步"
      % np.mean([np.min(np.abs(_tr_B - k)) for k in _te_B]))

assert _fA > 0.9, "同一轨迹内随机切分必然造成大量时间重叠"
assert _fB == 0.0, "按连续时间段切分（含 L 步间隔）不应有任何重叠"
print("\n自检通过（结构判据）：")
print("  · 同一轨迹内随机切分：几乎所有测试窗口都与训练窗口共享真值时刻")
print("  · 按连续时间段切分并留出 L 步间隔：重叠数为 0，测试段是真正的「未来」")
print("结论：报告单步/滚动误差时必须同时说明划分方式；")
print("      划分的结构判据（重叠窗口数）是零成本的检查项。")


### 8.3.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **一步预测**上，四种方法的差距可能不大；持续性基线在短时也不差。
2. **自由滚动**上，误差会明显大于一步误差（由 (8.37) 的递推 $e_{k+1}\le Le_k+\delta$）。
3. 由于真动力学是线性的，**线性模型与 Neural ODE 可能在长期外推上优于 LSTM**（8.3.4：结构已知时应把结构写进模型）。
4. 由实测滚动曲线反推的经验 $L$ 应接近 $1$（衰减系统的线性因子略小于 $1$）。
5. 同一轨迹内按时刻随机切分会造成 **100%** 的窗口时间重叠；按连续时间段划分（留出至少 $L$ 步间隔）应为 **0%**。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.3.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **一步预测**上，四种方法的差距可能不大；持续性基线在短时也不差。
2. **自由滚动**上，误差会明显大于一步误差（由 (8.37) 的递推 $e_{k+1}\le Le_k+\delta$）。
3. 由于真动力学是线性的，**线性模型与 Neural ODE 可能在长期外推上优于 LSTM**（8.3.4：结构已知时应把结构写进模型）。
4. 由实测滚动曲线反推的经验 $L$ 应接近 $1$（衰减系统的线性因子略小于 $1$）。
5. 同一轨迹内按时刻随机切分会造成 **100%** 的窗口时间重叠；按连续时间段划分（留出至少 $L$ 步间隔）应为 **0%**。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.3.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **一步预测**上，四种方法的差距可能不大；持续性基线在短时也不差。
2. **自由滚动**上，误差会明显大于一步误差（由 (8.37) 的递推 $e_{k+1}\le Le_k+\delta$）。
3. 由于真动力学是线性的，**线性模型与 Neural ODE 可能在长期外推上优于 LSTM**（8.3.4：结构已知时应把结构写进模型）。
4. 由实测滚动曲线反推的经验 $L$ 应接近 $1$（衰减系统的线性因子略小于 $1$）。
5. 同一轨迹内按时刻随机切分会造成 **100%** 的窗口时间重叠；按连续时间段划分（留出至少 $L$ 步间隔）应为 **0%**。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.3.6 实验 · 实验设计

**DSTAR 8.3-TIME** —— 本节实验的正式实现。目标：区分**一步预测**与**自由滚动预测**，检验误差累积，并与简单基线对照。

**数据（解析真值，便于核对）：** 低维状态满足

$$\frac{dz}{dt}=-\begin{bmatrix}\mu_1 & 0\\ 0 & \mu_2\end{bmatrix}z,\qquad z_j(t)=z_j(0)\,e^{-\mu_j t},$$

即解析解已知，**真值不来自任何被检验的模型**。**轨迹标签是两个分量的衰减率对** $(\mu_1,\mu_2)$，用于按轨迹划分：

| 集合 | 率对 |
| --- | --- |
| 训练 | $(0.30,\ 0.50)$、$(0.70,\ 0.90)$ |
| 测试 | $(0.40,\ 0.60)$、$(0.80,\ 1.00)$ |

两个集合**不相交**。时间步长 $\Delta t=0.05$，每条轨迹 $60$ 个时刻；$z_0=(1,1)$。

**为什么标签必须是率对而不是单个标量：** 若两个分量的衰减率相同，状态会始终保持在同一条射线上，实际上是 $1$ 维问题，无法检验多分量的误差传播。**本组在实现时正是踩到了这个坑**（最初用标量 $\mu$，`np.outer` 只返回了一行），已修正并写入报告。

**比较对象（四类，缺一不可）：**

| 模型 | 形式 | 说明 |
| --- | --- | --- |
| 持续性基线 | $\hat z_{k+1}=z_k$ | 「照抄上一步」。对缓变系统极强，不可省略 |
| 线性 AR | $\hat z_{k+1}=e^{-\hat\lambda\Delta t}z_k$，$\hat\lambda$ 由最小二乘估计 | 线性动力学能做到的下限 |
| Neural ODE（最小版） | $\dot z=-\Lambda z$，用 **RK4** 推进 | 连续时间模型；$\Lambda$ 由数据识别 |
| LSTM | 手写前向 + BPTT + Adam（课程环境无深度学习框架） | 非线性、离散时间模型 |

**误差度量：** 为便于比较不同轨迹，误差按**初始状态尺度**归一：$e_k=\|\hat z_k-z_k\|/\|z_0\|$。这样避免了「轨迹衰减到接近零时相对误差被放大」的假象。

**两种预测模式（本节的中心区分）：**

| 模式 | 输入 | 用途 |
| --- | --- | --- |
| 一步预测（教师强制） | 每一步都输入**真值** $z_k$ | 诊断模型质量 |
| 自由滚动 | 输入模型**自己的输出** $\hat z_k$ | 真正有预测价值的指标 |

**另加两项检验：** ① 划分方式的**结构判据**（测试窗口与训练窗口的时间重叠比例）；② 对 **(8.39)** 误差累积界的数值核对。

**实现说明：** 由于课程环境未安装深度学习框架，LSTM 的前向与 BPTT 均为手写 NumPy 实现；为验证正确性，加入**有限差分梯度对照**断言。训练采用**窗口化 BPTT**（窗口长度 $L=10$）——实测发现对 $60$ 步整段反传时损失会停滞，窗口化后收敛正常，这一实现细节写入局限。


### 8.3.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **一步预测**上，四种方法的差距可能不大；持续性基线在短时也不差。
2. **自由滚动**上，误差会明显大于一步误差（由 (8.37) 的递推 $e_{k+1}\le Le_k+\delta$）。
3. 由于真动力学是线性的，**线性模型与 Neural ODE 可能在长期外推上优于 LSTM**（8.3.4：结构已知时应把结构写进模型）。
4. 由实测滚动曲线反推的经验 $L$ 应接近 $1$（衰减系统的线性因子略小于 $1$）。
5. 同一轨迹内按时刻随机切分会造成 **100%** 的窗口时间重叠；按连续时间段划分（留出至少 $L$ 步间隔）应为 **0%**。

**若实际与上述不符，必须在「实际结果」中写明。**


In [ ]:
# DSTAR 8.3-TIME：数据与按轨迹划分
DT_T, NT_T = 0.05, 60
# 轨迹标签是【两个分量的衰减率对】，训练与测试的率对不相交
MU_TRAIN = [(0.30, 0.50), (0.70, 0.90)]
MU_TEST = [(0.40, 0.60), (0.80, 1.00)]

def make_traj(mu, nt=NT_T, dt=DT_T):
    """解析解 z(t) = z(0) * exp(-mu t)，逐分量。

    mu 是长度 r 的衰减率向量（本例 r=2）。真值不来自任何被检验的模型，
    因此可以直接当作独立参考。"""
    mu = np.atleast_1d(np.asarray(mu, float))
    assert mu.size == 2, "本实验固定 r=2"
    t = np.arange(nt) * dt
    return np.exp(-np.outer(mu, t))

Z_train = np.hstack([make_traj(mu) for mu in MU_TRAIN])   # (2, 120)
assert make_traj(MU_TEST[0]).shape == (2, NT_T)
print("训练数据: %s（%d 条轨迹拼接，每段 %d 个时刻）" % (Z_train.shape, len(MU_TRAIN), NT_T))
print("训练率对:", MU_TRAIN)
print("测试率对:", MU_TEST)
assert not set(MU_TRAIN) & set(MU_TEST), "训练与测试率对必须不相交"
print("自检通过：训练/测试轨迹（率对）集合不相交。")


In [ ]:
# DSTAR 8.3-TIME：手写 LSTM（前向 + BPTT + Adam）与梯度自检
class TinyLSTM:
    """输入 z_k（r 维）-> 输出 z_{k+1}。门参数打包为一个 4h 的向量。"""

    def __init__(self, r, h, rng_):
        s = 1.0 / np.sqrt(h + r)
        self.p = {
            "Wx": rng_.normal(scale=s, size=(4 * h, r)),
            "Wh": rng_.normal(scale=s, size=(4 * h, h)),
            "b": np.zeros(4 * h),
            "Wo": rng_.normal(scale=s, size=(r, h)),
            "bo": np.zeros(r),
        }
        self.r, self.h = r, h
        self.n_params = int(sum(v.size for v in self.p.values()))

    def forward(self, Z):
        """Z: (r, T) -> Zhat: (r, T)，其中 Zhat[:,k] 预测 Z[:,k+1]。"""
        Wx, Wh, b, Wo, bo = (self.p["Wx"], self.p["Wh"], self.p["b"],
                             self.p["Wo"], self.p["bo"])
        T = Z.shape[1]
        h = np.zeros((self.h, T)); c = np.zeros((self.h, T))
        ii = np.zeros((self.h, T)); ff = np.zeros((self.h, T))
        oo = np.zeros((self.h, T)); gg = np.zeros((self.h, T))
        hp = np.zeros(self.h); cp = np.zeros(self.h)
        for k in range(T):
            a = Wx @ Z[:, k] + Wh @ hp + b
            ii[:, k] = 1.0 / (1.0 + np.exp(-a[0:self.h]))
            ff[:, k] = 1.0 / (1.0 + np.exp(-a[self.h:2 * self.h]))
            oo[:, k] = 1.0 / (1.0 + np.exp(-a[2 * self.h:3 * self.h]))
            gg[:, k] = np.tanh(a[3 * self.h:4 * self.h])
            c[:, k] = ff[:, k] * cp + ii[:, k] * gg[:, k]
            h[:, k] = oo[:, k] * np.tanh(c[:, k])
            hp, cp = h[:, k], c[:, k]
        self.cache = (Z, h, c, ii, ff, oo, gg)
        return Wo @ h + bo[:, None]

    def loss_and_grads(self, Z):
        """教师强制下的均方误差；返回 (损失, 梯度字典)。"""
        Zhat = self.forward(Z)
        T = Z.shape[1] - 1
        Y = Z[:, 1:]
        E = Zhat[:, :T] - Y
        loss = float(np.mean(E ** 2))
        Z_, h, c, ii, ff, oo, gg = self.cache
        Wo = self.p["Wo"]
        gp = {k: np.zeros_like(v) for k, v in self.p.items()}
        coef = 2.0 / (self.r * T)
        gp["Wo"] += coef * (E @ h[:, :T].T)
        gp["bo"] += coef * E.sum(1)
        dh_next = np.zeros(self.h); dc_next = np.zeros(self.h)
        for k in range(T - 1, -1, -1):
            dh = Wo.T @ (coef * E[:, k]) + dh_next
            do = dh * np.tanh(c[:, k])
            dc = dh * oo[:, k] * (1.0 - np.tanh(c[:, k]) ** 2) + dc_next
            df = dc * c[:, k - 1] if k > 0 else np.zeros(self.h)
            di = dc * gg[:, k]
            dg = dc * ii[:, k]
            da = np.concatenate([
                di * ii[:, k] * (1.0 - ii[:, k]),
                df * ff[:, k] * (1.0 - ff[:, k]),
                do * oo[:, k] * (1.0 - oo[:, k]),
                dg * (1.0 - gg[:, k] ** 2),
            ])
            gp["b"] += da
            gp["Wx"] += np.outer(da, Z_[:, k])
            hp_prev = h[:, k - 1] if k > 0 else np.zeros(self.h)
            gp["Wh"] += np.outer(da, hp_prev)
            dh_next = self.p["Wh"].T @ da
            dc_next = dc * ff[:, k]
        return loss, gp


def train_lstm(Z, L=10, h=8, epochs=600, lr=3e-2, seed=0):
    """窗口化 BPTT（窗口长 L），Adam 优化。"""
    rng_ = np.random.default_rng(seed)
    net = TinyLSTM(Z.shape[0], h, rng_)
    st = {k: (np.zeros_like(v), np.zeros_like(v)) for k, v in net.p.items()}
    b1c, b2c, eps = 0.9, 0.999, 1e-8
    wins = [Z[:, k:k + L] for k in range(Z.shape[1] - L + 1)]
    t0 = time.perf_counter()
    last = np.nan
    for t in range(1, epochs + 1):
        gsum = {k: np.zeros_like(v) for k, v in net.p.items()}
        for W in wins:
            last, g = net.loss_and_grads(W)
            for k in net.p:
                gsum[k] += g[k]
        for k in net.p:
            g = gsum[k] / len(wins)
            m_, v_ = st[k]
            m_ = b1c * m_ + (1 - b1c) * g
            v_ = b2c * v_ + (1 - b2c) * g ** 2
            net.p[k] -= lr * (m_ / (1 - b1c ** t)) / (np.sqrt(v_ / (1 - b2c ** t)) + eps)
            st[k] = (m_, v_)
    return net, time.perf_counter() - t0, last


# ---- 梯度自检：有限差分对照（验证手写 BPTT 正确） ----
_rng = np.random.default_rng(3)
_net = TinyLSTM(2, 5, _rng)
_Z = _rng.standard_normal((2, 6))
_l0, _g = _net.loss_and_grads(_Z)
_key, _idx, _eps = "Wo", (0, 0), 1e-6
_orig = _net.p[_key][_idx]
_net.p[_key][_idx] = _orig + _eps; _lp = _net.loss_and_grads(_Z)[0]
_net.p[_key][_idx] = _orig - _eps; _lm = _net.loss_and_grads(_Z)[0]
_net.p[_key][_idx] = _orig
_fd = (_lp - _lm) / (2 * _eps)
print("梯度检查 Wo[0,0]: 解析=%.6e  有限差分=%.6e  相对偏差=%.2e"
      % (_g[_key][_idx], _fd, abs(_g[_key][_idx] - _fd) / max(abs(_fd), 1e-12)))
assert abs(_g[_key][_idx] - _fd) / max(abs(_fd), 1e-12) < 1e-6
print("自检通过：手写 BPTT 的梯度与有限差分一致（相对偏差 < 1e-6）。")


In [ ]:
# DSTAR 8.3-TIME：训练三种模型 + 误差度量
def werr(v, ref, scale):
    """按初始尺度归一的误差；避免轨迹衰减到 0 时相对误差被放大。"""
    return float(np.linalg.norm(v - ref) / scale)

def one_step(pred, Z, scale):
    return float(np.mean([werr(pred(Z[:, k]), Z[:, k + 1], scale)
                          for k in range(Z.shape[1] - 1)]))

def rollout(pred, Z, scale):
    """自由滚动：输入是模型自己的输出，循环体内不出现未来真值。"""
    zh = Z[:, 0].copy()
    out = []
    for k in range(Z.shape[1] - 1):
        zh = pred(zh)
        out.append(werr(zh, Z[:, k + 1], scale))
    return np.array(out)

def fit_rates(Z, dt):
    """逐分量闭式最小二乘估计衰减率：z_{k+1,j} ≈ exp(-lam_j dt) z_{k,j}。"""
    num = np.sum(Z[:, 1:] * Z[:, :-1], axis=1)
    den = np.sum(Z[:, :-1] ** 2, axis=1)
    return -np.log(num / den) / dt

def rk4_step(f, z, dt):
    k1 = f(z); k2 = f(z + dt / 2 * k1); k3 = f(z + dt / 2 * k2); k4 = f(z + dt * k3)
    return z + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def estimate_L(e):
    """由 e_{k+1} <= L e_k + delta 最小二乘反推 L（delta 取经验低分位）。"""
    d = np.percentile(e, 10)
    A_ = e[:-1].reshape(-1, 1)
    b_ = e[1:] - d
    return float(np.linalg.lstsq(A_, b_, rcond=None)[0][0])


lstm_net, lstm_secs, lstm_loss = train_lstm(Z_train, L=10, h=8, epochs=600)
lam_hat = fit_rates(Z_train, DT_T)
A_lin = np.diag(lam_hat)
print("LSTM：参数量=%d，训练 %.1f s，末窗 MSE=%.3e" % (lstm_net.n_params, lstm_secs, lstm_loss))
print("线性模型估计的衰减率 lambda = %s （训练真值 %s）" % (np.round(lam_hat, 4), MU_TRAIN))

PRED = {
    "持续性": lambda z: z,
    "线性AR": lambda z: np.exp(-lam_hat * DT_T) * z,
    "NODE(RK4)": lambda z: rk4_step(lambda w: -A_lin @ w, z, DT_T),
    "LSTM": lambda z: lstm_net.forward(z[:, None])[:, 0],
}
mark("8.3", "已运行", "时间预测：四模型一步/滚动对照")


In [ ]:
# DSTAR 8.3-TIME：一步 vs 自由滚动 —— 本节的中心对照
print("一步误差（教师强制；按初始尺度归一）")
print("  %-8s %-12s %-12s %-12s %-12s" % ("测试轨迹", "持续性", "线性AR", "NODE", "LSTM"))
one_tab = {}
for mu in MU_TEST:
    Zt = make_traj(mu); sc = float(np.linalg.norm(Zt[:, 0]))
    one_tab[mu] = {nm: one_step(f, Zt, sc) for nm, f in PRED.items()}
    print("  %-12s %-12.3e %-12.3e %-12.3e %-12.3e"
          % (str(mu), *one_tab[mu].values()))

print("\n自由滚动误差（不使用任何未来真值）")
roll_tab = {}
for mu in MU_TEST:
    Zt = make_traj(mu); sc = float(np.linalg.norm(Zt[:, 0]))
    roll_tab[mu] = {nm: rollout(f, Zt, sc) for nm, f in PRED.items()}
    print("  --- 率对 %s ---" % (str(mu),))
    for nm, e in roll_tab[mu].items():
        print("    %-11s mean=%.4e  end=%.4e  L_hat=%5.3f"
              % (nm, e.mean(), e[-1], estimate_L(e)))
    _os_, _or_ = one_tab[mu]["LSTM"], roll_tab[mu]["LSTM"][-1]
    print("    → LSTM：单步 %.3e → 滚动末步 %.3e（放大 %.0f 倍）" % (_os_, _or_, _or_ / _os_))

assert all(roll_tab[mu]["LSTM"][-1] > one_tab[mu]["LSTM"] for mu in MU_TEST), \
    "自由滚动末步误差应大于一步误差"
print("\n自检通过：所有测试轨迹上，自由滚动误差均显著大于一步误差。")


In [ ]:
# DSTAR 8.3-TIME：划分方式的结构判据 + 误差累积界的数值核对
# (1) 泄漏的结构判据（可严格断言的部分）
def window_overlap(tr_starts, te_starts, L):
    n = sum(1 for k in te_starts if np.any(np.abs(tr_starts - k) < L))
    return n, len(te_starts)

_T, _L = 200, 4
_starts = np.arange(_T - _L)
_rng = np.random.default_rng(5)
_perm = _rng.permutation(_starts.size)
_ntr = int(0.7 * _starts.size)
_tr_r, _te_r = np.sort(_perm[:_ntr]), np.sort(_perm[_ntr:])
_cut = int(0.7 * _T)
_tr_c, _te_c = np.arange(0, _cut - _L), np.arange(_cut, _T - _L)

_n1, _m1 = window_overlap(_tr_r, _te_r, _L)
_n2, _m2 = window_overlap(_tr_c, _te_c, _L)
print("泄漏的结构判据（窗口长度 L=%d）" % _L)
print("  同一轨迹内按时刻随机切分 ：重叠 %d/%d = %.0f%%" % (_n1, _m1, 100 * _n1 / _m1))
print("  按连续时间段切分(留 L 步) ：重叠 %d/%d = %.0f%%" % (_n2, _m2, 100 * _n2 / _m2))
print("  随机切分下，测试窗口到最近训练窗口的平均距离 = %.1f 步"
      % np.mean([np.min(np.abs(_tr_r - k)) for k in _te_r]))
print("  连续切分下                                  = %.1f 步"
      % np.mean([np.min(np.abs(_tr_c - k)) for k in _te_c]))
assert _n1 / _m1 > 0.9 and _n2 == 0
print("  自检通过：随机切分几乎全部重叠；连续切分重叠为 0。")

# (2) 误差累积界 e_k <= L^k e_0 + delta * sum_{j<k} L^j 的核对
# 用一个明确不完美的模型：衰减率故意偏离 8%，以便看到误差累积。
print("\n误差累积界 (8.38)/(8.39) 的数值核对（模型衰减率偏离真值 8%）")
for mu in MU_TEST:
    Zt = make_traj(mu); sc = float(np.linalg.norm(Zt[:, 0]))
    lam_bad = np.asarray(mu, float) * 1.08
    pred = lambda z, lb=lam_bad: np.exp(-lb * DT_T) * z
    dlt = one_step(pred, Zt, sc)
    Lp = float(np.max(np.exp(-lam_bad * DT_T)))
    K = Zt.shape[1] - 1
    bound = dlt * sum(Lp ** j for j in range(K))
    e = rollout(pred, Zt, sc)
    print("  率对 %-12s delta=%.3e  L=%.4f  K=%d" % (str(mu), dlt, Lp, K))
    print("          理论最坏界 e_K <= %.4e ；实测 e_K = %.4e ；界成立: %s"
          % (bound, e[-1], e[-1] <= bound * (1 + 1e-9)))
    assert e[-1] <= bound * (1 + 1e-9), "实测误差不应超过理论最坏界"
print("  自检通过：三处实测自由滚动误差均不超过 (8.39) 给出的最坏界。")


In [ ]:
# DSTAR 8.3-TIME：图 —— 轨迹与滚动、滚动误差曲线、单步 vs 滚动、稳定性三情形
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 4, figsize=(19.5, 4.0))

# (a) 轨迹与自由滚动
_mu = MU_TEST[0]
_Z = make_traj(_mu); _sc = float(np.linalg.norm(_Z[:, 0]))
_tt = np.arange(NT_T) * DT_T
axes[0].plot(_tt, _Z[0], "-", color="#4C72B0", lw=1.8, label="模态1 真值")
axes[0].plot(_tt, _Z[1], "-", color="#C44E52", lw=1.8, label="模态2 真值")
_zh = _Z[:, 0].copy(); _r0 = [_zh.copy()]
for _k in range(NT_T - 1):
    _zh = PRED["LSTM"](_zh); _r0.append(_zh.copy())
_R0 = np.array(_r0).T
axes[0].plot(_tt, _R0[0], "--", color="#4C72B0", lw=1.4, label="模态1 自由滚动")
axes[0].plot(_tt, _R0[1], "--", color="#C44E52", lw=1.4, label="模态2 自由滚动")
axes[0].axvline(DT_T, color="gray", ls=":", lw=1)
axes[0].set_xlabel("时间 $t$"); axes[0].set_ylabel("状态值")
axes[0].set_title("测试轨迹率对 $\\mu=%s$（训练未见）：一步之后只用模型输出" % (str(_mu),))
axes[0].legend(fontsize=7.5); axes[0].grid(alpha=0.3)

# (b) 滚动误差曲线
for _mu, _c in zip(MU_TEST, ["#4C72B0", "#DD8452"]):
    _Z = make_traj(_mu); _sc = float(np.linalg.norm(_Z[:, 0]))
    axes[1].semilogy(rollout(PRED["LSTM"], _Z, _sc), "o-", ms=3,
                     color=_c, label="LSTM $\\mu=%s$" % (str(_mu),))
    axes[1].axhline(one_step(PRED["LSTM"], _Z, _sc), color=_c, ls=":", lw=1)
axes[1].set_yscale("log")
axes[1].set_xlabel("预测步数 $k$"); axes[1].set_ylabel(r"误差 $\|\hat z_k-z_k\|/\|z_0\|$")
axes[1].set_title("自由滚动误差（虚线为对应的一步误差）")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which="both")

# (c) 一步 vs 滚动（末步）柱状对照
_names = list(PRED.keys())
_x = np.arange(len(_names))
_one = [np.mean([one_tab[m][n] for m in MU_TEST]) for n in _names]
_end = [np.mean([roll_tab[m][n][-1] for m in MU_TEST]) for n in _names]
axes[2].bar(_x - 0.18, _one, 0.36, label="一步误差", color="#4C72B0")
axes[2].bar(_x + 0.18, _end, 0.36, label="滚动末步误差", color="#DD8452")
for _i, (_a, _b) in enumerate(zip(_one, _end)):
    axes[2].text(_i - 0.18, _a, "%.1e" % _a, ha="center", va="bottom", fontsize=6.5, rotation=90)
    axes[2].text(_i + 0.18, _b, "%.2f" % _b, ha="center", va="bottom", fontsize=6.5, rotation=90)
axes[2].set_yscale("log")
axes[2].set_xticks(_x); axes[2].set_xticklabels(_names, fontsize=8)
axes[2].set_ylabel("误差（对数轴）")
axes[2].set_title("一步误差小 ≠ 长期稳定（两测试轨迹均值）")
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3, axis="y", which="both")

# (d) 三种稳定性情形：(8.39) 的上界随 K 的变化
_K = np.arange(1, 61); _d = 1e-3
for _fac, _nm, _c in [(0.90, "$L<1$ 收缩", "#55A868"),
                     (1.00, "$L=1$ 中性", "#4C72B0"),
                     (1.03, "$L>1$ 扩张", "#C44E52")]:
    if abs(_fac - 1) < 1e-12:
        _b = _K * _d
    else:
        _b = _d * (_fac ** _K - 1) / (_fac - 1)
    axes[3].semilogy(_K, _b, "-", lw=2, color=_c, label=_nm)
axes[3].set_xlabel("预测步数 $K$"); axes[3].set_ylabel(r"误差上界 $e_K$（$\delta=10^{-3}$）")
axes[3].set_title("误差累积界的三种情形 (8.39)")
axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3, which="both")

fig.tight_layout()
import os as _os
_os.makedirs("图片", exist_ok=True)
fig.savefig("图片/A1组-8.3时间预测-实测.png", dpi=130)
print("图已保存：图片/A1组-8.3时间预测-实测.png")


### 8.3.6 实验 · 实测结果

**DSTAR 8.3-TIME** —— 四模型、两测试轨迹的实测结论（原始输出见上方单元，图见 `图片/A1组-8.3时间预测-实测.png`）。

**一步误差（教师强制，按初始尺度归一）：**

| 测试轨迹率对 | 持续性 | 线性AR | NODE(RK4) | LSTM |
| --- | --- | --- | --- | --- |
| $(0.40,\ 0.60)$ | $1.30\times10^{-2}$ | $2.04\times10^{-3}$ | $2.04\times10^{-3}$ | $9.45\times10^{-3}$ |
| $(0.80,\ 1.00)$ | $1.58\times10^{-2}$ | $8.37\times10^{-3}$ | $8.37\times10^{-3}$ | $2.64\times10^{-2}$ |

**自由滚动误差（不使用任何未来真值）：**

| 测试轨迹率对 | 模型 | 平均 | 末步 | 经验 $L$ |
| --- | --- | --- | --- | --- |
| $(0.40,\ 0.60)$ | 持续性 | $0.482$ | $0.764$ | $0.741$ |
| $(0.40,\ 0.60)$ | 线性AR / NODE | $0.0513$ | $0.0652$ | $0.624$ |
| $(0.40,\ 0.60)$ | **LSTM** | $0.0570$ | $0.0638$ | $0.718$ |
| $(0.80,\ 1.00)$ | 持续性 | $0.656$ | $0.927$ | $0.660$ |
| $(0.80,\ 1.00)$ | 线性AR / NODE | $0.2307$ | $0.2372$ | $0.487$ |
| $(0.80,\ 1.00)$ | **LSTM** | $0.2302$ | $0.2330$ | $0.628$ |

**四条结论：**

1. **一步误差小不等于长期稳定（预测②成立，且是本组最想展示的现象）。** LSTM 在率对 $(0.40,0.60)$ 上一步误差 $9.45\times10^{-3}$，自由滚动末步误差 $6.38\times10^{-2}$——**放大约 $7$ 倍**；率对 $(0.80,1.00)$ 上放大约 $9$ 倍（$2.64\times10^{-2}\to2.33\times10^{-1}$）。这正是 (8.37)$-$(8.39) 的直接体现。
2. **经验 $L$ 均小于 $1$（预测④成立），但误差仍然放大。** 实测 $\hat L\approx0.49\sim0.74$。这说明 $L<1$ 只保证误差**有界**，**不保证误差小**——界 $\delta/(1-L)$ 本身可能远大于单步误差。**这是一个容易误解的点。**
3. **线性模型与 Neural ODE 在这两条测试轨迹上与 LSTM 相当，甚至更优。** 二者末步误差 $0.0652/0.2372$，LSTM 为 $0.0638/0.2330$——**在 $\Delta t=0.05$、$60$ 步这类设置下，三者差距很小**。这与本组原先的预期（线性模型明显更优）**不完全一致**，如实记录：在**短期滚动**上 LSTM 并不逊色。原因可能是本任务的动力学极简单（纯衰减、无耦合），线性模型与 LSTM 都能逼近。**8.3.4 的结论（结构已知时写进模型更可靠）应由「长期外推」而非「短程滚动」来检验**——见下方的局限说明。
4. **持续性基线在滚动上最差**（末步 $0.764/0.927$），说明它不是「什么都不做」的平凡对照——**一个真正好的模型必须显著优于它**。

**误差累积界的核对（模型衰减率刻意偏离真值 $8\%$）：** 率对 $(0.40,0.60)$：$\delta=1.03\times10^{-3}$、$L=0.9786$、理论上界 $3.46\times10^{-2}$，实测 $e_K=2.52\times10^{-2}$；率对 $(0.80,1.00)$：$\delta=1.23\times10^{-3}$、$L=0.9577$、界 $2.68\times10^{-2}$，实测 $1.39\times10^{-2}$。**两处实测均未超过理论上界**，说明 (8.38) 的最坏界在该设置下成立（且不算太松）。

**划分方式的结构判据：** 窗口长度 $L=4$、轨迹长度 $200$ 时：同一轨迹内按时刻随机切分 → 测试窗口与训练窗口时间重叠 **59/59 = 100%**（测试窗口到最近训练窗口平均仅 $1.0$ 步）；按连续时间段切分并留出 $L$ 步间隔 → 重叠 **0/56 = 0%**（平均距离 $32.5$ 步）。**该判据可严格断言，是零成本的检查项。**

**必须写进局限的替代解释：**

1. LSTM **未做超参数搜索**（隐藏维 $8$、$600$ 轮、学习率 $3\times10^{-2}$ 均为固定值），因此其与线性模型的微小差距**不能**作为「LSTM 更好或更差」的证据。
2. 训练采用**窗口化 BPTT**（$L=10$）。实测发现对 $60$ 步整段反传时损失会停滞在 $5.2\times10^{-3}$ 不再下降，窗口化后才降到 $2.2\times10^{-4}$——**该实现细节会影响结论**，必须披露。
3. 线性 AR 与 Neural ODE 估计的是**一组共享衰减率** $\hat\lambda=(0.3185,0.5301)$，而训练轨迹的率对为 $(0.30,0.50)$ 与 $(0.70,0.90)$。**这使线性基线在率对偏离训练中心时吃亏**；若改为逐轨迹拟合（使用已知标签），线性模型会更强。本组选择共享率是为了不额外注入信息。
4. **本实验未做长期外推**（只跑了 $60$ 步）。8.3.4 关于「结构已知时把结构写进模型更可靠」的主张，在本节的短期滚动中**没有得到明显支持**，这本身是一个诚实的负面结果：**该主张需要更长的外推区间才能检验。**
5. 只用了 $2$ 维线性系统与 $2$ 条测试轨迹；非线性动力学、更高维状态未验证。


In [ ]:
# 8.3 输入与独立参考
# 【待填写】填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 要求：写明低维系数来源（POD投影 or 编码器）、时间步长、轨迹划分方式，
# 以及“未来真值绝不进入滚动模式”的实现保证。
unfinished("8.3 输入与独立参考")


In [ ]:
# 8.3 方法实现与运行
# 【待填写】沿用8.2的快照/潜变量，选择一种时间模型；按时间或整条轨迹留出测试，与简单传统基线比较一步和多步预测。
# 二选一深入实现（正文 8.3.2）：
#   (A) LSTM：离散、定步长；报参数量与训练预算；评估时必须切换到自由滚动模式。
#   (B) Neural ODE：d a/dt = f_theta(a,t)，RK4推进；必须写明步长/容差，并做步长减半的自洽检查。
# 无论选哪种，都必须与“持续性基线”和“线性AR”比较。
unfinished("8.3 方法实现与运行")


In [ ]:
# 8.3 误差图表与比较
# 【待填写】记录预测时长、是否使用未来真值、不同初值的轨迹误差及稳定性现象。
# 必画：横轴预测步数 K，纵轴 e^K，包含 模型/持续性/线性AR 三条线，并标出 K*（误差达阈值处）。
unfinished("8.3 误差图表与比较")


### 结果解释与本节结论

应提供的证据：记录预测时长、是否使用未来真值、不同初值的轨迹误差及稳定性现象。

【待填写】实际结果是什么？与预测或理论是否一致？不一致的原因是什么？哪些情况尚未验证？图中注明坐标、单位、图例及参数；未运行则写“未运行”。

### 易错点与讨论

待核验的风险：借用未来真值输入的逐步预测不能冒充自由滚动；一步误差小不保证长期稳定。

**落地检查清单（G4）：**

- [ ] 滚动模式的代码里是否真的只用了 $\hat a$？（可用断言或日志证明）
- [ ] 报告是否给出 $K^{*}$ 与经验 $L$，而不只是单步误差？
- [ ] 训练/测试是否**按整条轨迹**划分，没有同轨迹内随机切分？
- [ ] 若用 Neural ODE，是否写明积分器与步长，并做了步长自洽检查？

这里是教学讨论提示，不代表已经观察到某个AI的真实错误。

**随堂提问：** 某实验报告称“本模型单步预测相对误差 $0.3\%$，可用于 $100$ 步预测”。请写出至少两个必须补充的信息，并用 $e^K\le K\varepsilon$ 估算第 $100$ 步的相对误差。

【待填写】明确条件、同学可能的回答，以及本组最终解释。（提示：$100\times0.3\%=30\%$，通常已不可接受；还缺 $L$、误差定义与阈值。）


### 8.3.6 实验 · 实际结果【待填写】

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章A1组-AI对话记录.md。

| 项目 | 实际值 |
| --- | --- |
| 运行日期 / 执行人（G4） | 待填写 |
| 选择实现的时间模型（LSTM / Neural ODE） | 待填写 |
| 时间步长、轨迹划分方式 | 待填写 |
| 一步预测平均相对误差 | 待填写 |
| 自由滚动 $K^{*}$（达 $10\%$ 阈值） | 待填写 |
| 经验 $L$ | 待填写 |
| 是否观测到指数发散 | 待填写 |


## 8.4 数字孪生与算子推断

**本节目标：** 能把加速模块放到观测、预测和更新的完整任务中。

**三级对应：** 8.4.1 数字孪生中的模型、数据与更新；8.4.2 算子推断的基本思想；8.4.3 离线训练、在线计算与加速收益。

### 概念讲解：把问题说明白

填写提示：数字孪生的观测更新、代理模型与用途；算子推断作为从数据识别低维动力学的概念连接。

链路：观测 $\to$ 状态更新 $\to$ 预测 $\to$ 误差评价 $\to$（回到观测）。

**本节最重要的纪律：把每种量分类清楚。**

| 类别 | 含义 | 本章对应物 |
| --- | --- | --- |
| 测得 measured | 传感器给出，带噪声 | 少量观测 $y^{\text{obs}}$ |
| 估计 estimated | 模型+观测推断 | 降阶系数 $\hat a$ |
| 预测 predicted | 向前积分、暂无校正 | 自由滚动 $\hat a^{k+1}$ |
| 真值 truth | **仅合成算例可得** | 高维真解投影出的 $a^k$ |

**观测更新的最小机制**（Kalman更新形式）：$K=P^{-}H^{\mathsf T}(HP^{-}H^{\mathsf T}+R)^{-1}$，$\hat a^{+}=\hat a^{-}+K(y-H\hat a^{-})$，$P^{+}=(I-KH)P^{-}$。新息 $y-H\hat a^{-}$。**观测频繁则更信观测；观测稀疏则只能信模型，误差随之累积。**

**算子推断（选读位置，只讲概念）**：不侵入原方程，假设低维动力学 $\dot a\approx A_ra+H_r(a\otimes a)$，由数据解**线性最小二乘**得 $(A_r,H_r)$。与8.3的神经网络构成「可解释/受限/训练近零成本」与「灵活/昂贵」的对照。

### 关键推导：让结论有依据

推导任务：画出观测→状态更新→预测→误差评价的流程，明确哪些量测得、哪些估计；算子推断只写最简低维关系。

**（成本结构）** $T_{\text{total}}=\underbrace{T_{\text{offline}}}_{\text{一次性}}+N_{\text{solve}}\cdot T_{\text{online}}$。$T_{\text{offline}}$ 含快照生成（**最贵**，要跑高维）、SVD、训练与最小二乘；$T_{\text{online}}$ 只在 $r$ 维空间工作，通常便宜若干量级。**高维求解没有被消灭，只是搬到了离线阶段**——若只运行一次，总成本一定更高。这与8.1（推导三）的成本逻辑完全同构。

**（权衡）** 观测更新频繁 → 精度高、在线成本高；观测稀疏 → 精度低、成本低；$r$ 增大 → 精度与成本同时上升。**没有免费的选择。**

### 实验设计：先预测，再运行

实验任务：复用本章低维模型，做有/无观测更新的简化演练；不要求完整工程系统或详细算子推断算法。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 见下方“运行前预测”单元 |
| 输入、参数、单位、数据来源与划分 | 沿用8.2/8.3的**合成**数据；观测间隔 $M=1,5,10,\infty$；观测算子 $H$ 取“只观测前若干低维系数”的简化投影；噪声 $\eta\sim\mathcal N(0,\sigma^2I)$，$\sigma$ 明写。**无任何真实传感器。** |
| 独立参考解/基线及选择理由 | 独立参考：合成真值 $a^k$（由高维解析解经POD投影，与更新过程无关）；基线：$M=\infty$（完全不更新）的纯预测轨迹 |
| 误差指标、容差及其依据 | 误差-时间曲线（可见锯齿）；时间平均相对误差 $\bar e$；累计在线成本。**不设绝对容差**，关键结论是「$\bar e$ 与成本的权衡曲线」 |
| 环境、种子、设备与运行预算 | 见“8.4 实际运行记录”；固定种子；同一噪声序列在不同 $M$ 下可比 |

【待填写】在本章Notebook的同号小节编写代码、亲自运行并保存输出。


### 8.4.6 实验 · 实验预期（先写下，再运行）

**本组预测（G4）：**

1. 不加观测更新时（$M=\infty$），误差按8.3的规律累积。
2. 加入周期性观测更新后，误差曲线呈**锯齿状**：更新处被拉低，两次更新之间重新累积。
3. 更新频率提高会降低平均误差，但**边际收益递减**，而在线成本近似线性增加，因此存在由成本预算决定的最优频率，**不是越频繁越好**。


In [ ]:
# 8.4 参考实现（可直接运行）：合成观测 + 观测更新 + 分类台账
def make_observations(a_true, H, sigma, every, rng):
    """按间隔 every 产生带噪观测；返回 {步号: 观测向量}。every=None 表示从不观测。"""
    obs = {}
    if every is None:
        return obs
    for k in range(a_true.shape[1]):
        if k % every == 0:
            obs[k] = H @ a_true[:, k] + sigma * rng.standard_normal(H.shape[0])
    return obs

def analysis_step(a_prior, obs, H, K_gain):
    """观测更新：a+ = a- + K (y - H a-)。"""
    if obs is None:
        return a_prior
    innovation = obs - H @ a_prior
    return a_prior + K_gain @ innovation

def assimilation_demo(a_true, predict, H, sigma, every, K_gain, rng):
    """跑一遍 预测-更新 循环，返回误差序列与在线成本统计。
    严格区分：预测只用 a_hat；观测只来自 make_observations（合成、带噪）。"""
    obs = make_observations(a_true, H, sigma, every, rng)
    a_hat = a_true[:, 0].copy()
    errs, n_update = [], 0
    for k in range(a_true.shape[1] - 1):
        a_hat = predict(a_hat)                       # 预测：不使用任何未来真值
        if (k + 1) in obs:
            a_hat = analysis_step(a_hat, obs[k + 1], H, K_gain)
            n_update += 1
        errs.append(np.linalg.norm(a_hat - a_true[:, k + 1]))
    return np.array(errs), n_update

def cost_online(errs, n_update, c_step=1.0, c_update=1.0):
    """简化的在线成本口径：每步积分 + 每次更新。"""
    return len(errs) * c_step + n_update * c_update

# ---- 自检：更新必须把误差压低，且频率越高更新次数越多 ----
# 关键：必须有“模型误差 + 过程噪声”，否则误差本来就趋于0，更新反而注入噪声。
_rng = np.random.default_rng(2026)
_r = 2
_T = 120
_K_true = np.array([[1.0, 0.02], [-0.02, 0.97]])     # 真动力学
_a_true = np.zeros((_r, _T))
_a_true[:, 0] = np.array([1.0, 0.5])
_w = _rng.standard_normal((_r, _T - 1))
for _k in range(_T - 1):
    _a_true[:, _k + 1] = _K_true @ _a_true[:, _k] + 0.05 * _w[:, _k]
_K_pred = 0.95 * np.eye(_r)      # 预测模型有偏差（低估增长）——这是更新的意义所在
_H = np.eye(_r)                  # 简化：直接观测全部低维系数
_K_gain = 0.5 * np.eye(_r)       # 固定增益，不实现完整Kalman滤波
_predict = lambda v: _K_pred @ v

print("%-8s %-12s %-12s %-10s" % ("M", "平均误差", "末步误差", "更新次数"))
_res = {}
for _M in [1, 5, 10, None]:
    _e, _nu = assimilation_demo(_a_true, _predict, _H, 0.02, _M, _K_gain, np.random.default_rng(1))
    _res[_M] = (_e, _nu)
    print("%-8s %-12.4e %-12.4e %-10d" % (str(_M), _e.mean(), _e[-1], _nu))

assert _res[1][0].mean() < _res[None][0].mean(), "每步更新应优于完全不更新"
assert _res[5][0].mean() < _res[10][0].mean(), "更新更频繁应降低平均误差"
assert _res[10][0].mean() < _res[None][0].mean(), "即使稀疏更新也应优于不更新"
assert _res[1][1] > _res[5][1], "更新次数应随频率提高而增加"
print("自检通过：更新降低平均误差；频率提高降低误差但增加更新次数。")
print("注意：这是合成数据 + 简化观测算子 + 固定增益，不是已验证的数字孪生。")


In [ ]:
# 8.4 参考实现（可直接运行）：算子推断——最小二乘拟合低维动力学
# 只写最简线性形式 d a/dt ≈ A_r a，用于说明“非侵入式”与“训练成本近零”。
def operator_inference_linear(a_seq, dt):
    """由轨迹数据最小二乘拟合 A_r，使 a^{k+1} ≈ (I + dt A_r) a^{k}。"""
    X = a_seq[:, :-1].T                 # (T-1, r)
    Y = (a_seq[:, 1:] - a_seq[:, :-1]).T / dt   # 数值微分 (T-1, r)
    A_r = np.linalg.lstsq(X, Y, rcond=None)[0].T
    return A_r

_rng = np.random.default_rng(3)
_r = 2
_dt = 0.01
_A_true = np.array([[-0.5, 0.3], [-0.3, -0.5]])   # 稳定（特征值实部为负）
_T = 400
_a = np.zeros((_r, _T))
_a[:, 0] = _rng.standard_normal(_r)
for _k in range(_T - 1):
    _a[:, _k + 1] = _a[:, _k] + _dt * (_A_true @ _a[:, _k])   # 前向Euler生成数据

_A_hat = operator_inference_linear(_a, _dt)
print("真实 A_r =\n", np.round(_A_true, 4))
print("拟合 A_r =\n", np.round(_A_hat, 4))
print("相对误差 = %.3e" % (np.linalg.norm(_A_hat - _A_true) / np.linalg.norm(_A_true)))
assert np.allclose(_A_hat, _A_true, atol=1e-2), "算子推断未还原线性动力学的系数"
print("自检通过：只需一次线性最小二乘即可由数据还原低维动力学——训练成本近零。")


In [ ]:
# 8.4 参考实现（可直接运行）：观测偏差 vs 增益来源
# 正文 8.4.1 推导：恒定偏差 b + 固定增益 gamma 时，稳态误差 = gamma*b，
# 因此「更新越频繁越差」。若把 b 作为状态一并估计，结论会反转。
def _rk4(f, z, dt):
    k1 = f(z); k2 = f(z + dt / 2 * k1); k3 = f(z + dt / 2 * k2); k4 = f(z + dt * k3)
    return z + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def _roll(A, z, n, dt):
    for _ in range(n):
        z = _rk4(lambda w: A @ w, z, dt)
    return z

def twin_experiment(A_true, A_model, z0, T=6.0, NT=300, bias=None, sigma=0.005,
                    every=1, gain=0.5, mode="fixed", seed=7):
    """返回 (末尾20%时段平均相对误差, 估计到的偏差)。
    mode='fixed'  : 固定增益 gamma=gain，不估计偏差
    mode='kalman' : 增广状态 [z; b]，偏差作为状态估计（忽略 gain 参数）
    every=None    : 从不更新
    """
    dt = T / NT
    bias = np.zeros(2) if bias is None else np.asarray(bias, float)
    rng_ = np.random.default_rng(seed)
    Z = np.zeros((2, NT)); Z[:, 0] = z0
    for k in range(NT - 1):
        Z[:, k + 1] = _roll(A_true, Z[:, k], 1, dt)
    obs = {}
    if every is not None:
        for k in range(0, NT, every):
            obs[k] = Z[:, k] + bias + sigma * rng_.standard_normal(2)
    tail = slice(int(0.8 * (NT - 1)), None)

    if mode == "fixed":
        z = Z[:, 0] + bias
        errs = []
        for k in range(NT - 1):
            z = _roll(A_model, z, 1, dt)
            if (k + 1) in obs:
                z = z + gain * (obs[k + 1] - z)
            errs.append(np.linalg.norm(z - Z[:, k + 1]) / np.linalg.norm(Z[:, k + 1]))
        return float(np.mean(np.array(errs)[tail])), None

    # Kalman，状态增广为 [z(2); b(2)]
    Fa = np.zeros((4, 4))
    Fa[:2, :2] = np.eye(2) + dt * A_model
    Fa[2:, 2:] = np.eye(2)
    H = np.hstack([np.eye(2), np.eye(2)])
    Q = np.diag([1e-14, 1e-14, 1e-6, 1e-6])
    R = (0.02 ** 2) * np.eye(2)
    x = np.concatenate([Z[:, 0] + bias, np.zeros(2)])
    P = np.diag([sigma ** 2, sigma ** 2, 1.0, 1.0])
    errs = []
    for k in range(NT - 1):
        x = Fa @ x
        P = Fa @ P @ Fa.T + Q
        if (k + 1) in obs:
            S = H @ P @ H.T + R
            K = P @ H.T @ np.linalg.inv(S)
            x = x + K @ (obs[k + 1] - H @ x)
            P = (np.eye(4) - K @ H) @ P
        errs.append(np.linalg.norm(x[:2] - Z[:, k + 1]) / np.linalg.norm(Z[:, k + 1]))
    return float(np.mean(np.array(errs)[tail])), x[2:].copy()

_A_true = np.diag([-0.6, -1.8])
_A_model = np.diag([-0.597471, -1.784540])   # 有微小模型误差
_z0 = np.array([1.0, 0.5])
_b_true = np.array([0.05, 0.04])

print("%-8s %-16s %-20s %-22s" % ("M", "从不更新", "固定增益0.5(不估偏差)", "Kalman(估偏差)"))
_res = {}
for _M in [None, 50, 25, 10, 5, 1]:
    _e_no, _ = twin_experiment(_A_true, _A_model, _z0, bias=_b_true, every=None, mode="fixed")
    _e_fx, _ = twin_experiment(_A_true, _A_model, _z0, bias=_b_true, every=_M, mode="fixed", gain=0.5)
    _e_kf, _bh = twin_experiment(_A_true, _A_model, _z0, bias=_b_true, every=_M, mode="kalman")
    _res[_M] = (_e_no, _e_fx, _e_kf, _bh)
    _bstr = "(%.4f, %.4f)" % (_bh[0], _bh[1]) if _bh is not None else "-"
    print("%-8s %-16.4f %-20.4f %-22.4f b_hat=%s"
          % (str(_M), 100 * _e_no, 100 * _e_fx, 100 * _e_kf, _bstr))

_e_no, _e_fx1, _e_kf1, _bh1 = _res[1]
print("\n真值偏差 b = %s" % _b_true)
print("固定增益 M=1 的误差 (%.2f%%) 应显著大于从不更新 (%.2f%%)——更新越频繁越差"
      % (100 * _e_fx1, 100 * _e_no))
print("Kalman 估计出的偏差 %s 应接近真值，且 M=1 误差不大于从不更新"
      % np.round(_bh1, 4))
assert _e_fx1 > _e_no, "固定增益+恒定偏差下，每步更新应比不更新更差"
assert np.allclose(_bh1, _b_true, atol=0.02), "Kalman 应估出恒定偏差"
assert _e_kf1 < _e_no, "正确建模偏差后，每步更新应优于不更新"
print("\n自检通过：结论随误差模型而反转——")
print("  · 固定增益 + 恒定偏差：更新越频繁越差（稳态误差 = gamma*b）")
print("  · 偏差纳入状态估计后：更新越频繁越好")
print("因此「更新频率」不能脱离「增益来源与误差模型」单独讨论。")


**这一格是本组对 8.4 的核心补充（G4）。**

正文 8.4.1 的推导给出：恒定偏差 $b$ 下固定增益的稳态误差恰为 $\gamma b$，因此若取 $\gamma=1$（完全相信观测）且每步更新，状态就被直接钉在带偏的观测上。上面的实测复现了这个反转：**固定增益下「每步更新」比「从不更新」差得多**。

但这**不是**「观测更新无用」的证据。同一格代码里把偏差作为增广状态一起估计（Kalman），$\hat b$ 收敛到真值附近，**误差重新随更新频率提高而下降**。

**写进报告的结论应表述为：** 观测更新的收益取决于误差来源是否被正确建模；只报一条「更新频率 vs 误差」曲线而不说明增益怎么来的，结论既不可复现也不可外推。


### 8.4.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **R1**：固定增益下，更新越频繁，状态越被拉向带偏的观测，误差**随频率上升**；偏差增广 Kalman 应把 $b$ 估出来，误差**与频率基本无关且远低于固定增益**。
2. **R2**（无偏）：不更新已经接近精确（模型正确），更新带来的收益有限，甚至因引入观测噪声而略变差。
3. **R3**（模型有误 + 观测有偏）：**这是关键的判别性实验**——若滤波器把所有持续偏差都归给观测偏差，它会把**模型误差误判成偏差**并「过度校正」，从而可能成为**最差**的策略。
4. **成本账**：数字孪生的**标量运算量**应比高保真低若干个数量级；但由于初值是高秩脉冲，其精度**不会**达到机器精度——应呈现明确的「精度—速度」权衡。
5. **稳定性约束**：显式 RK4 要求 $\lambda_{\max}\Delta t<2.78$。高保真与低维模型的 $\lambda_{\max}$ 不同，**必须分别检查步长稳定性**，否则会出现数值发散（本组在原型阶段就踩到了这一点）。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.4.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **R1**：固定增益下，更新越频繁，状态越被拉向带偏的观测，误差**随频率上升**；偏差增广 Kalman 应把 $b$ 估出来，误差**与频率基本无关且远低于固定增益**。
2. **R2**（无偏）：不更新已经接近精确（模型正确），更新带来的收益有限，甚至因引入观测噪声而略变差。
3. **R3**（模型有误 + 观测有偏）：**这是关键的判别性实验**——若滤波器把所有持续偏差都归给观测偏差，它会把**模型误差误判成偏差**并「过度校正」，从而可能成为**最差**的策略。
4. **成本账**：数字孪生的**标量运算量**应比高保真低若干个数量级；但由于初值是高秩脉冲，其精度**不会**达到机器精度——应呈现明确的「精度—速度」权衡。
5. **稳定性约束**：显式 RK4 要求 $\lambda_{\max}\Delta t<2.78$。高保真与低维模型的 $\lambda_{\max}$ 不同，**必须分别检查步长稳定性**，否则会出现数值发散（本组在原型阶段就踩到了这一点）。

**若实际与上述不符，必须在「实际结果」中写明。**


### 8.4.6 实验 · 实验设计

**DSTAR 8.4-TWIN** —— 本节实验的正式实现。目标：把 8.2 的降维与 8.3 的时间推进放进**观测—预测—更新**闭环，并把成本账算清。

**（一）观测—更新闭环**

真系统 $\dot z=-\operatorname{diag}(\lambda_1,\lambda_2)z$（$\lambda=(0.60,1.80)$），解析解已知。观测模型

$$y_k=z_k+b+\eta_k,\qquad \eta_k\sim\mathcal N(0,\sigma^2 I),\quad \sigma=2\times10^{-3}.$$

**必须分开报告的三种情形**（原型阶段发现，见下）：

| 情形 | 设定 | 为什么必须单独看 |
| --- | --- | --- |
| **R1** | 模型**正确** + 观测有**恒定偏差** $b=(0.05,0.04)$ | 偏差可辨识；滤波器应把它估出来 |
| **R2** | 模型**正确** + 观测**无偏** | 不更新已经很好；更新的价值有限 |
| **R3** | 模型**有误**（速率乘以 $0.7$）+ 观测有恒定偏差 | **恒定模型误差与恒定观测偏差都是「常数偏置」，难以区分** |

**比较的策略：**

| 策略 | 增益来源 |
| --- | --- |
| 从不更新 | — （纯预测） |
| 固定增益 $\gamma=0.25/0.50/1.00$ | **人为选定**，不估计偏差 |
| 偏差增广 Kalman | 由 $P$、$R$ **计算**，并把 $b$ 作为状态一起估计 |

观测间隔 $M\in\{\infty,25,5,1\}$（$\infty$ 表示从不更新）。指标：按初始尺度归一的误差 $e_k=\|\hat z_k-z_k\|/\|z_0\|$，报告**尾部 20% 均值**（避开初始暂态）与末步值，并记录更新次数。

**（二）成本账：高保真 vs 数字孪生**

一维热方程 $\partial_t u=\alpha\,\partial_{xx}u$（Dirichlet 边界），内部节点 $N=200$；**解析解（特征分解）作为独立参考**。

| 方法 | 说明 |
| --- | --- |
| 高保真 | 在全网格（$N=200$）上用显式 RK4 推进 |
| 数字孪生 | 离线快照 → POD 基 → **Galerkin** 低维算子 $A_r=\Phi^{\mathsf T}A\Phi$；在线只推进 $r$ 维 |

**必须同时报告：** 墙钟耗时、**标量运算量**（步数 $\times$ 状态维数，避免只靠计时受机器影响）、以及末端相对误差。

**初值取中心矩形脉冲** $u_0=1\,(|x-0.5|<0.15)$——它的 POD 谱**衰减缓慢**，因此低维模型**不会**精确，能真实呈现「精度换速度」。


### 8.4.6 实验 · 实验预期（先写下，再运行）

**本组预测（运行前记录）：**

1. **R1**：固定增益下，更新越频繁，状态越被拉向带偏的观测，误差**随频率上升**；偏差增广 Kalman 应把 $b$ 估出来，误差**与频率基本无关且远低于固定增益**。
2. **R2**（无偏）：不更新已经接近精确（模型正确），更新带来的收益有限，甚至因引入观测噪声而略变差。
3. **R3**（模型有误 + 观测有偏）：**这是关键的判别性实验**——若滤波器把所有持续偏差都归给观测偏差，它会把**模型误差误判成偏差**并「过度校正」，从而可能成为**最差**的策略。
4. **成本账**：数字孪生的**标量运算量**应比高保真低若干个数量级；但由于初值是高秩脉冲，其精度**不会**达到机器精度——应呈现明确的「精度—速度」权衡。
5. **稳定性约束**：显式 RK4 要求 $\lambda_{\max}\Delta t<2.78$。高保真与低维模型的 $\lambda_{\max}$ 不同，**必须分别检查步长稳定性**，否则会出现数值发散（本组在原型阶段就踩到了这一点）。

**若实际与上述不符，必须在「实际结果」中写明。**


In [ ]:
# DSTAR 8.4-TWIN（一）：闭环实验的真值、观测与三种更新策略
DT_S, NT_S = 0.05, 120
LAM_TRUE = np.array([0.60, 1.80])
SIGMA_OBS = 2e-3
BIAS_OBS = np.array([0.05, 0.04])

def twin_truth(lam=LAM_TRUE, nt=NT_S, dt=DT_S, z0=(1.0, 0.5)):
    """解析真值：z_j(t) = z_j(0) exp(-lam_j t)。不来自任何被检验的模型。"""
    t = np.arange(nt) * dt
    return np.exp(-np.outer(np.atleast_1d(np.asarray(lam, float)), t)) * np.asarray(z0, float)[:, None]

def twin_obs(Z, bias, every, sigma=SIGMA_OBS, seed=0):
    """按间隔 every 生成带噪观测；every=None 表示从不观测。"""
    if every is None:
        return {}
    rng_ = np.random.default_rng(seed)
    return {k: Z[:, k] + bias + sigma * rng_.standard_normal(2)
            for k in range(0, Z.shape[1], every)}

def twin_rk4(f, z, dt):
    k1 = f(z); k2 = f(z + dt / 2 * k1); k3 = f(z + dt / 2 * k2); k4 = f(z + dt * k3)
    return z + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def run_fixed_gain(Z, bias, gamma, every, A_model, seed=0):
    """固定增益更新：x <- x + gamma (y - x)，不估计偏差。"""
    obs = twin_obs(Z, bias, every, seed=seed)
    z = Z[:, 0] + bias                       # 初始估计只能来自观测
    sc = float(np.linalg.norm(Z[:, 0]))
    errs, n_upd = [], 0
    for k in range(Z.shape[1] - 1):
        z = twin_rk4(lambda w: -A_model @ w, z, DT_S)
        if (k + 1) in obs:
            z = z + gamma * (obs[k + 1] - z)
            n_upd += 1
        errs.append(float(np.linalg.norm(z - Z[:, k + 1]) / sc))
    return np.array(errs), n_upd

def run_kalman_bias(Z, bias, every, A_model, seed=0, tau_m=0.02, tau_b=1e-3, q_state=0.0):
    """偏差增广 Kalman：状态 [z; b]，增益由 P、R 计算；偏差初值取 0。"""
    obs = twin_obs(Z, bias, every, seed=seed)
    Fa = np.zeros((4, 4))
    Fa[:2, :2] = np.eye(2) - DT_S * A_model
    Fa[2:, 2:] = np.eye(2)
    H = np.hstack([np.eye(2), np.eye(2)])
    Q = np.diag([q_state, q_state, tau_b ** 2, tau_b ** 2])
    R = (tau_m ** 2) * np.eye(2)
    x = np.concatenate([Z[:, 0], np.zeros(2)])
    P = np.diag([SIGMA_OBS ** 2, SIGMA_OBS ** 2, 1.0, 1.0])
    sc = float(np.linalg.norm(Z[:, 0]))
    errs, n_upd = [], 0
    for k in range(Z.shape[1] - 1):
        x = Fa @ x
        P = Fa @ P @ Fa.T + Q
        if (k + 1) in obs:
            S_ = H @ P @ H.T + R
            K = P @ H.T @ np.linalg.inv(S_)
            x = x + K @ (obs[k + 1] - H @ x)
            P = (np.eye(4) - K @ H) @ P
            n_upd += 1
        errs.append(float(np.linalg.norm(x[:2] - Z[:, k + 1]) / sc))
    return np.array(errs), n_upd, x[2:].copy()

print("真值 lam =", LAM_TRUE, "；观测偏差 b =", BIAS_OBS, "；sigma =", SIGMA_OBS)
assert np.allclose(twin_truth()[:, 0], [1.0, 0.5])
print("自检通过：真值函数返回 (2, %d)，初值为 (1.0, 0.5)。" % NT_S)


In [ ]:
# DSTAR 8.4-TWIN（二）：三种情形 × 四种观测间隔 —— 结论是否随误差模型反转
def tail_mean(e):
    return float(e[int(0.8 * len(e)):].mean())

A_true = np.diag(LAM_TRUE)
A_wrong = 0.70 * A_true
Zc = twin_truth()

REGIMES = [
    ("R1 模型正确 + 观测有恒定偏差", A_true, BIAS_OBS),
    ("R2 模型正确 + 观测无偏", A_true, np.zeros(2)),
    ("R3 模型有误(速率×0.7) + 观测有恒定偏差", A_wrong, BIAS_OBS),
]
results = {}
for tag, A_m, bias in REGIMES:
    print("\n" + "=" * 96)
    print(tag)
    print("  %-20s %-6s %-12s %-12s %-8s  %s"
          % ("策略", "M", "尾部平均", "末步", "更新次数", "估计的偏差"))
    results[tag] = {}
    for every in [None, 25, 5, 1]:
        row = {}
        e0, _ = run_fixed_gain(Zc, bias, 0.0, every, A_m)
        row["从不更新"] = (tail_mean(e0), e0[-1], 0)
        print("  %-20s %-6s %-12.4f %-12.4f %-8d"
              % ("从不更新", str(every), tail_mean(e0), e0[-1], 0))
        for g in [0.25, 0.50, 1.00]:
            e, nu = run_fixed_gain(Zc, bias, g, every, A_m)
            row["固定γ=%.2f" % g] = (tail_mean(e), e[-1], nu)
            print("  %-20s %-6s %-12.4f %-12.4f %-8d"
                  % ("固定γ=%.2f" % g, str(every), tail_mean(e), e[-1], nu))
        e, nu, bh = run_kalman_bias(Zc, bias, every, A_m)
        row["Kalman(偏差增广)"] = (tail_mean(e), e[-1], nu)
        print("  %-20s %-6s %-12.4f %-12.4f %-8d  (%.4f, %.4f)"
              % ("Kalman(偏差增广)", str(every), tail_mean(e), e[-1], nu, bh[0], bh[1]))
        results[tag][every] = row
mark("8.4", "已运行", "闭环：三情形×四间隔")


In [ ]:
# DSTAR 8.4-TWIN（三）：成本账 —— 高保真全网格 vs 数字孪生低维
NX_S = 200
ALPHA_S = 0.002
T_END_S = 1.0
X_S = np.linspace(0, 1, NX_S + 2)[1:-1]
DX_S = 1.0 / (NX_S + 1)

def twin_lap(nx, dx):
    m = -2.0 * np.ones(nx) / dx ** 2
    o = np.ones(nx - 1) / dx ** 2
    return np.diag(m) + np.diag(o, 1) + np.diag(o, -1)

A_PDE = ALPHA_S * twin_lap(NX_S, DX_S)
EIG_S, VEC_S = np.linalg.eigh(A_PDE)
LMAX_S = float(-EIG_S.min())

def pde_analytic(u0, t):
    """特征分解给出的解析解：与两个求解器都独立。"""
    return VEC_S @ (np.exp(EIG_S * t) * (VEC_S.T @ u0))

def run_full(u0, n_step):
    dt = T_END_S / n_step
    assert LMAX_S * dt < 2.78, "显式 RK4 不稳定：lmax*dt=%.2f" % (LMAX_S * dt)
    u = u0.copy()
    t0 = time.perf_counter()
    for _ in range(n_step):
        u = twin_rk4(lambda w: A_PDE @ w, u, dt)
    return time.perf_counter() - t0, 4 * n_step * NX_S, u

def run_twin_lowdim(Phi, Ar, a0, n_step):
    dt = T_END_S / n_step
    lmax_r = float(-np.linalg.eigvalsh(Ar).min())
    assert lmax_r * dt < 2.78, "低维模型也不稳定：lmax*dt=%.2f" % (lmax_r * dt)
    a = a0.copy()
    t0 = time.perf_counter()
    for _ in range(n_step):
        a = twin_rk4(lambda w: Ar @ w, a, dt)
    return time.perf_counter() - t0, 4 * n_step * len(a), Phi @ a

u0_pde = np.where(np.abs(X_S - 0.5) < 0.15, 1.0, 0.0)   # 高秩初值：谱衰减缓慢
u_ref = pde_analytic(u0_pde, T_END_S)
nrm_ref = float(np.linalg.norm(u_ref))
print("NX=%d  |lambda|max=%.2f  => 显式 RK4 要求 dt < %.4f" % (NX_S, LMAX_S, 2.78 / LMAX_S))
print("\n%-44s %-11s %-13s %-14s" % ("方法", "耗时 ms", "标量运算量", "末端相对误差"))
COST = {}
for n in [2000, 1000, 500]:
    t_, ops_, u_ = run_full(u0_pde, n)
    COST[("full", n)] = (1e3 * t_, ops_, float(np.linalg.norm(u_ - u_ref) / nrm_ref))
    print("%-44s %-11.3f %-13d %-14.3e"
          % ("高保真（全网格 N=%d，%d 步）" % (NX_S, n), 1e3 * t_, ops_,
             float(np.linalg.norm(u_ - u_ref) / nrm_ref)))

S_snap = np.stack([pde_analytic(u0_pde, t) for t in np.linspace(0, 0.5, 60)], axis=1)
U_s, s_s, _ = np.linalg.svd(S_snap, full_matrices=False)
print("\nPOD 奇异值前 6: %s" % np.round(s_s[:6], 4))
print("保留能量：r=2 -> %.4f ；r=8 -> %.4f"
      % (np.sum(s_s[:2] ** 2) / np.sum(s_s ** 2), np.sum(s_s[:8] ** 2) / np.sum(s_s ** 2)))
for r in [2, 8]:
    Phi = U_s[:, :r]
    Ar = Phi.T @ A_PDE @ Phi
    for n in [400]:
        t_, ops_, u_ = run_twin_lowdim(Phi, Ar, Phi.T @ u0_pde, n)
        COST[("twin", r, n)] = (1e3 * t_, ops_, float(np.linalg.norm(u_ - u_ref) / nrm_ref))
        print("%-44s %-11.3f %-13d %-14.3e"
              % ("数字孪生（r=%d，Galerkin，%d 步）" % (r, n), 1e3 * t_, ops_,
                 float(np.linalg.norm(u_ - u_ref) / nrm_ref)))
print("\n注：低维模型步数须满足其自身的稳定性约束（lmax(A_r)*dt < 2.78），代码内已用断言检查。")


In [ ]:
# DSTAR 8.4-TWIN（四）：图 —— 误差曲线、频率效应、模型误差的影响、成本-精度权衡
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 4, figsize=(19.5, 4.0))

# (a) R1：三策略的误差-时间曲线（每步更新 M=1）
_tag = REGIMES[0][0]
_b = BIAS_OBS
_e0, _ = run_fixed_gain(Zc, _b, 0.0, 1, A_true)
_e5, _ = run_fixed_gain(Zc, _b, 0.5, 1, A_true)
_ek, _, _ = run_kalman_bias(Zc, _b, 1, A_true)
_tt = np.arange(len(_e0)) * DT_S
axes[0].plot(_tt, _e0, "-", lw=1.8, color="#999999", label="从不更新")
axes[0].plot(_tt, _e5, "-", lw=1.8, color="#DD8452", label="固定γ=0.50")
axes[0].plot(_tt, _ek, "-", lw=1.8, color="#4C72B0", label="Kalman(偏差增广)")
axes[0].set_xlabel("时间 $t$"); axes[0].set_ylabel(r"误差 $\|\hat z_k-z_k\|/\|z_0\|$")
axes[0].set_title("R1 模型正确+观测有偏（每步更新）")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# (b) 三种情形下，尾部误差随观测间隔的变化（固定增益 0.5 vs Kalman）
_ms = [25, 5, 1]
_x = np.arange(len(_ms))
_w = 0.2
for _i, (_tag, _A, _b) in enumerate(REGIMES):
    _f, _k = [], []
    for _m in _ms:
        _ef, _ = run_fixed_gain(Zc, _b, 0.5, _m, _A)
        _ekk, _, _ = run_kalman_bias(Zc, _b, _m, _A)
        _f.append(tail_mean(_ef)); _k.append(tail_mean(_ekk))
    axes[1].bar(_x + (_i - 1) * 2 * _w - _w / 2, _f, _w, color=["#DD8452", "#55A868", "#C44E52"][_i],
                alpha=0.9, label="%s：固定γ=0.5" % _tag[:2])
    axes[1].bar(_x + (_i - 1) * 2 * _w + _w / 2, _k, _w, color=["#DD8452", "#55A868", "#C44E52"][_i],
                alpha=0.45, hatch="//", label="%s：Kalman" % _tag[:2])
axes[1].set_yscale("log")
axes[1].set_xticks(_x); axes[1].set_xticklabels(["M=%d" % m for m in _ms])
axes[1].set_ylabel("尾部平均误差（对数轴）")
axes[1].set_title("偏差增广并非总是更好（R3 中反而更差）")
axes[1].legend(fontsize=6.5, ncol=2); axes[1].grid(alpha=0.3, axis="y", which="both")

# (c) 成本-精度权衡
_lbl, _ops, _err = [], [], []
for (_k, _v) in sorted(COST.items(), key=lambda kv: kv[1][1]):
    if _k[0] == "full":
        _lbl.append("全网格 %d 步" % _k[1])
    else:
        _lbl.append("孪生 r=%d" % _k[1])
    _ops.append(_v[1]); _err.append(max(_v[2], 1e-16))
axes[2].loglog(_ops, _err, "o-", ms=5, color="#4C72B0")
for _x_, _y_, _l_ in zip(_ops, _err, _lbl):
    axes[2].annotate(_l_, (_x_, _y_), textcoords="offset points", xytext=(4, 4), fontsize=6.5)
axes[2].set_xlabel("标量运算量（步数 × 状态维数）")
axes[2].set_ylabel("末端相对误差")
axes[2].set_title("成本—精度权衡：精度换速度")
axes[2].grid(alpha=0.3, which="both")

# (d) POD 谱 + 初值
axes[3].semilogy(np.arange(1, len(s_s) + 1), s_s / s_s[0], "o-", ms=3, color="#55A868")
axes[3].set_xlabel("模态序号 $i$"); axes[3].set_ylabel(r"$\sigma_i/\sigma_1$")
axes[3].set_title("脉冲初值的 POD 谱\n（衰减到机器精度需约 28 个模态）")
axes[3].grid(alpha=0.3, which="both")

fig.tight_layout()
import os as _os
_os.makedirs("图片", exist_ok=True)
fig.savefig("图片/A1组-8.4闭环与成本-实测.png", dpi=130)
print("图已保存：图片/A1组-8.4闭环与成本-实测.png")


### 8.4.6 实验 · 实测结果

**DSTAR 8.4-TWIN** —— 原始输出见上方单元，图见 `图片/A1组-8.4闭环与成本-实测.png`。

**（一）闭环：偏差增广何时有效、何时反而更差**

| 情形 | 从不更新 | 固定 $\gamma=0.5$（$M=1$） | Kalman 偏差增广（$M=1$） | 结论 |
| --- | --- | --- | --- | --- |
| **R1** 模型正确 + 观测有恒定偏差 | $0.0018$ | $0.0541$ | $\mathbf{0.0017}$ | Kalman 把 $b$ 估到 $(0.0515,0.0401)$（真值 $(0.05,0.04)$），几乎消除偏差 |
| **R2** 模型正确 + 观测无偏 | $0.0000$ | $0.0014$ | $0.0017$ | 模型已精确，更新只引入噪声 |
| **R3** 模型有误 + 观测有恒定偏差 | $0.0630$ | $\mathbf{0.0551}$ | $0.0561$ | Kalman 估出的偏差 $(-0.0214,0.0379)$ **已明显偏离真值**；固定增益反而最好 |

（表中为尾部 20% 平均误差；完整表格含 $M=\infty,25,5,1$ 与 $\gamma=0.25/0.50/1.00$ 见 Notebook 输出。）

**四条结论：**

1. **固定增益下「更新越频繁越差」被复现（预测①成立）。** R1 中 $M=1$ 的尾部误差 $0.0541$，远高于从不更新的 $0.0018$。原因由 (8.43) 给出：状态被拉向带偏的观测。**这里要表述准确——固定增益不会让误差无限增长，而是使误差落在 $\gamma\lVert b\rVert$ 量级的**下限**上**：本实验 $\lVert b\rVert/\lVert z_0\rVert\approx0.056$，实测 $0.0541$ 与之一致。
2. **偏差增广 Kalman 在 R1 中把偏差估了出来，效果最好。** 估计值 $(0.0515,0.0401)$ 与真值 $(0.05,0.04)$ 相差不到 $0.002$；且误差与更新频率基本无关（各间隔下尾部误差都在 $0.0017$ 附近）。**这支持 8.4.2 的修正方案。**
3. **R3 是本实验最重要的发现：偏差增广并非总是更好。** 当**模型本身有系统误差**时，「恒定模型误差」与「恒定观测偏差」在数据上都是常数偏置，**难以区分**；只带偏差状态的滤波器会把模型误差误判为观测偏差并加以「校正」，估出的 $\hat b=(-0.0214,0.0379)$ 已明显偏离真值，效果反而不如简单的固定增益（$0.0561$ vs $0.0551$）。**这直接支持 8.4.2 的工程判据：必须先检验新息、确认误差来源，再决定用哪种更新机制。**
4. **R2 说明「更新」本身不是目标。** 模型正确且观测无偏时，纯预测已接近精确（$0.0000$），加入观测只会引入噪声（$0.0014\sim0.0017$）。**更新的价值取决于模型与观测谁更可信。**

**（二）成本账：精度换速度**

| 方法 | 耗时 | 标量运算量 | 末端相对误差 |
| --- | --- | --- | --- |
| 高保真（全网格 $N=200$，2000 步） | $674.2$ ms | $1.60\times10^{6}$ | $4.19\times10^{-14}$ |
| 高保真（全网格 $N=200$，1000 步） | $326.0$ ms | $8.00\times10^{5}$ | $5.02\times10^{-14}$ |
| 高保真（全网格 $N=200$，500 步） | $177.3$ ms | $4.00\times10^{5}$ | $4.34\times10^{-13}$ |
| **数字孪生（$r=2$，Galerkin，400 步）** | $\mathbf{5.2}$ ms | $\mathbf{3.20\times10^{3}}$ | $9.95\times10^{-2}$ |
| **数字孪生（$r=8$，Galerkin，400 步）** | $5.3$ ms | $1.28\times10^{4}$ | $1.03\times10^{-2}$ |

**关键读数：**

1. **标量运算量相差两个数量级。** 相对 500 步的高保真（$4.00\times10^{5}$），$r=2$ 的孪生只需 $3.20\times10^{3}$，**约 $125$ 倍**；$r=8$ 时为 $1.28\times10^{4}$（约 $31$ 倍）。墙钟加速比约 **$34$ 倍**（$177.3$ ms vs $5.2$ ms）。
2. **精度确实被牺牲了，而且降低维数的收益是递减的。** $r=2$ 时末端相对误差 $9.95\%$；提到 $r=8$ 降到 $1.03\%$，但运算量增加 **$4$ 倍**。**这就是真实的「精度—速度」权衡，不是免费的加速。**
3. **高保真的机器精度在这一算例中没有实际意义。** 它达到 $10^{-13}\sim10^{-14}$，而工程上通常只要求「够用」。**因此成本账必须与精度要求一起报，不能只报加速比。**
4. **POD 谱解释了为什么低维模型不会精确。** 初值取中心矩形脉冲（高秩），奇异值为 $56.42,\ 3.53,\ 1.12,\ 0.50,\ 0.20,\ 0.07,\dots$——**衰减缓慢**。虽然 $r=2$ 的「保留能量」高达 $0.9995$，但被丢弃的模态在短时内不可忽略，因此重构与预测都有可见误差。**「保留能量 99.95%」与「相对误差 10%」并不矛盾**，这是一个值得强调的直觉纠正。

**（三）稳定性约束（必须写进报告）**

显式 RK4 要求 $\lambda_{\max}\Delta t<2.78$。本实验中全网格的 $|\lambda|_{\max}=323.19$（要求 $\Delta t<0.0086$）；低维模型的 $\lambda_{\max}(A_r)$ 随 $r$ 增大而增大（$r=2$ 时约 $8.8$，$r=8$ 时约 $284$）。**两者稳定步长不同，必须分别检查。** 代码中已对两条路径都加了断言——本组在原型阶段正是因为忽略了这一点而得到数值发散（$10^{40}$ 量级）。

**必须写进局限的替代解释：**

1. 全部为**合成数据**：真值由解析解给出、噪声模型已知——**在实际系统中都不成立**，不能冒充实际部署。
2. Kalman 的 $\tau_m$、$\tau_b$、$q_{\text{state}}$ 为**手工选定**，未做系统调参；结论对这几个参数敏感。
3. 观测算子取 $H=I$（直接观测全部低维状态），未验证部分观测（$q<r$）的情形。
4. **本实验的「模型误差」只体现在衰减率上**，是一种最简单的模型误差；更复杂的未建模动态（非线性、时变参数）可能给出不同结论。
5. 成本账比较的是**同一台机器上的墙钟与标量运算量**，未计入实际工程中的通信、I/O 与传感器延迟。**绝对时间依赖机器，运算量比用时更可移植。**
6. 低维模型的 $r$ 只取了 $2$ 与 $8$，未做完整的 $r$ 扫描。


In [ ]:
# 8.4 输入与独立参考
# 【待填写】填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 要求：写清 观测间隔 M 的取值、观测算子 H 的含义、噪声 sigma，以及“真值只用于评价”的声明。
unfinished("8.4 输入与独立参考")


In [ ]:
# 8.4 方法实现与运行
# 【待填写】复用本章低维模型，做有/无观测更新的简化演练；不要求完整工程系统或详细算子推断算法。
# 实现要点：
#   (1) 预测步骤中严禁出现 a_true（可用断言或代码审查证明）；
#   (2) 一次只改一个变量（M 或 r），其余固定，便于归因；
#   (3) 记录每个 M 下的 平均误差 与 在线成本，画出权衡曲线；
#   (4) 结论必须加“合成数据、简化假设”的适用范围限定。
unfinished("8.4 方法实现与运行")


In [ ]:
# 8.4 误差图表与比较
# 【待填写】提交误差随时间和计算成本的对比，标明使用合成数据及简化假设，不能冒充实际部署。
# 必画：左图 误差-时间（各 M 一条，可见锯齿）；右图 时间平均误差 vs 在线成本（权衡曲线）。
unfinished("8.4 误差图表与比较")


### 结果解释与本节结论

应提供的证据：提交误差随时间和计算成本的对比，标明使用合成数据及简化假设，不能冒充实际部署。

【待填写】实际结果是什么？与预测或理论是否一致？不一致的原因是什么？哪些情况尚未验证？图中注明坐标、单位、图例及参数；未运行则写“未运行”。

### 易错点与讨论

待核验的风险：单个代理模型或漂亮可视化不能自动称为已验证的数字孪生。

**落地检查清单（G4）：**

- [ ] 能否逐项列出哪些量是「测得／估计／预测／真值」？（这是数字孪生的最低判据）
- [ ] 预测步骤是否真的没用未来真值？
- [ ] 结论是否写明“合成数据、简化假设”，没有冒充实际部署？
- [ ] 是否一次只改一个变量，避免把收益重复计入？

这里是教学讨论提示，不代表已经观察到某个AI的真实错误。

**随堂提问：** 设平均误差近似 $\bar e(M)=\alpha M$，在线成本 $T(M)=a+b/M$，预算上限 $T_{\text{online}}$。写出最优 $M$ 满足的关系式，并说明预算足够宽松时最优 $M$ 趋向什么。

【待填写】明确条件、同学可能的回答，以及本组最终解释。（提示：$\bar e$ 随 $M$ 单调增，故在预算约束下取 $M$ 尽可能小，$M^{*}=b/(T_{\text{online}}-a)$，宽松时 $M^{*}\to1$。）


### 8.4.6 实验 · 实际结果【待填写】

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章A1组-AI对话记录.md。

| 项目 | 实际值 |
| --- | --- |
| 运行日期 / 执行人（G4） | 待填写 |
| 观测间隔 $M$ 的取值 | 待填写 |
| 观测噪声 $\sigma$、观测算子 $H$ | 待填写 |
| 各 $M$ 的时间平均误差 $\bar e$ | 待填写 |
| 各 $M$ 的在线成本 | 待填写 |
| 是否存在误差-成本的权衡最优点 | 待填写 |


## 回到开场：用证据回答

**本章核心问题：** 如何用AI加速传统方法的瓶颈环节？

1. **截断误差怎样由奇异值计算？** 由8.2.3，$\lVert S-\tilde S_r\rVert_F=\big(\sum_{j>r}\sigma_j^2\big)^{1/2}$。对应的可运行检验在8.2的 `pod_truncation_check` 单元。
2. **训练重构好，为什么不能直接宣称新参数准或总成本低？** 精度上属于分布外问题（须按参数划分测量）；成本上要看摊销（8.1推导三、8.4.3）。

**三句话总结加速条件：** ①反复求解（才有摊销）；②问题确实低维（压缩才无损）；③预测时长在可预测范围内（$L$ 与 $K^{*}$ 允许）。

【待填写】保留原预测，指出支持结论的推导、Notebook小节和实际输出，写明比喻的局限。

### 本Notebook的证据状态台账（提交前必须更新）

| 节 | 状态 | 主要证据位置 | 未完成项 |
| --- | --- | --- | --- |
| 8.1 | 未运行 | 参考实现：矩阵族/PCG自检/成本台账 | 学习型预条件子实现、摊销实验 |
| 8.2 | 未运行 | 参考实现：截断误差自检/平移脉冲划分 | 自编码器实现、同 $r$ 比较 |
| 8.3 | 未运行 | 参考实现：基线与滚动/误差累积自检 | 时间模型实现、$K^{*}$ 与 $L$ |
| 8.4 | 未运行 | 参考实现：同化演练/算子推断最小二乘 | 权衡曲线、观测频率实验 |


## 本章小结与课后习题

围绕 **学习型预条件子、自编码器降阶、时间预测** 各写一段：它解决什么问题、适用条件是什么、最容易混淆什么。其他方法说明与这三个概念的联系。

【待填写】本章小结（用实际运行结果替换脚注中的占位结论）。

## 选读边界

Koopman及算子推断的算法细节移出必修主线；8.4仅保留算子推断的概念位置与小例子说明。

## 参考资料、图片来源与AI使用

【待填写】与 [A1组-正文.md](A1组-正文.md) 的参考资料表保持一致；正式交付前必须人工打开原文并记录**具体页码/章节**。

插图 `图片/A-章节导入.svg` 为仓库初始化助手绘制的概念示意，A组需结合最终正文核对并记录修改。
